# TabTune on the Lexsi platform · Home Credit · Loan Amount Recommendation
### The same ten-section flow as the open-source notebook, run on Lexsi endpoints through the SDK

**How much credit should this applicant be offered?** A retail lender recommends a loan amount from the applicant's profile, bureau history and previous applications, to support limit setting and affordability checks.

| | |
|---|---|
| **Task** | Regression: credit amount (`AMT_CREDIT`) |
| **Engine** | Lexsi platform: managed TabTune builds on platform compute, called through `lexsi-sdk` |
| **In this notebook** | Data preparation and plotting only. Training, scoring, explanations, guardrails, drift dashboards, monitors and the LLM gateway all run on the platform |
| **Agents** | Data-prep agent, retraining agent and model-card writer. Each one calls an LLM through the **Lexsi gateway**, and every call is traced |

## Notebook map

| # | Section | On the platform |
|---|---|---|
| 1 | Onboarding | Login, project, **endpoint map** (SDK call → REST endpoint), **LLM gateway** with several models |
| 2 | Product options | Problem statement, data tags, what runs where |
| 3 | Data preparation | Profile dashboards, **LLM-reviewed data-prep agent** with human gates, upload by tag, platform data checks |
| 4 | Model training | Zero-shot foundation models, ensemble, leaderboard and distillation builds; selection; global SHAP / LIME |
| 5 | Model testing | `model_inference` on the Test tag, evaluation dashboards |
| 6 | Batch inference | `model_inference` on a production tag |
| 7 | Model risk | Uncertainty, fairness, **observations and policies**, regulatory evidence map |
| 8 | Artefacts | Model card written through the gateway with automatic checks; one case with SHAP, LIME, observations and policies |
| 9 | Model in production | **Drift dashboards**, **monitors and alerts**, a signed webhook and the **LLM-reviewed retraining agent** |
| 10 | Agent traces and conclusions | Every LLM call per agent, read back from the gateway (sessions → messages → traces); audit trail |

**How to read it:** short visible cells are the platform calls. Collapsed cells (▸ *Show code*) hold helpers and plotting; their output still shows.

**Before you run:** a Lexsi access token for `lexsi.login()`; a Colab secret `LEXSI_API_TOKEN` for the gateway's REST route (used if the gateway project cannot be opened through the SDK); the Home Credit files in Google Drive. No GPU is needed in Colab: the models run on platform compute. Every platform call is wrapped, so a call your account cannot run prints the platform's message and the notebook carries on.

# 1 · Onboarding

## 1.1 Connect

In [ ]:
# @title Install the Lexsi SDK (pinned) and plotting packages { display-mode: "form" }
import os, sys, subprocess
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lexsi-sdk==0.1.69", "plotly", "ipywidgets", "httpx[http2]"], capture_output=True, text=True)
if r.returncode != 0:
    print((r.stdout + r.stderr)[-3000:]); raise RuntimeError("pip install failed; pip's message is printed above")
from importlib.metadata import version
print("lexsi-sdk", version("lexsi-sdk"))

In [ ]:
from lexsi_sdk import lexsi
lexsi.login()                      # paste your Lexsi access token when asked

In [ ]:
# @title Imports, theme, fact sheet, audit trail and approval gates { display-mode: "form" }
import json, logging, re, time, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown, HTML
import ipywidgets as W

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
pd.set_option("display.max_columns", 60)

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    GPU = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "none"
except ImportError:
    DEVICE, GPU = "cpu", "none"

FAST = os.environ.get("TABTUNE_DEMO_FAST") == "1"      # tiny settings for a smoke test
AUTO_APPROVE = True                                    # False = the agents stop and ask you to type yes / no
BLUE, ORANGE, TEAL, GREY, DARK, RED, GOLD = "#2F5BEA", "#B4541A", "#1B998B", "#9AA0A8", "#16181D", "#D7263D", "#E8C547"
PALETTE = [BLUE, ORANGE, TEAL, RED, GOLD, GREY, DARK, "#7A5CFA"]
FACTS = {}                                             # every number that reaches the model card is recorded here
AUDIT = []                                             # every agent step and human decision


def record(section, **values):
    """Store results for the model card and the conclusions."""
    FACTS.setdefault(section, {}).update({k: (round(float(v), 4) if isinstance(v, (int, float, np.floating, np.integer)) and not isinstance(v, bool) else v)
                                          for k, v in values.items()})


def audit(actor, step, finding, decision=""):
    AUDIT.append({"at": pd.Timestamp.now().strftime("%H:%M:%S"), "actor": actor, "step": step, "finding": finding, "decision": decision})
    print(f"[{actor} · {step}] {finding}" + (f"  →  {decision}" if decision else ""))


def human_gate(question, recommended=True):
    """Human-in-the-loop checkpoint. With AUTO_APPROVE the recommendation is taken and logged; otherwise you are asked."""
    if AUTO_APPROVE:
        answer = recommended
        who = "auto-approved (AUTO_APPROVE=True)"
    else:
        reply = input(f"🧑 {question}  [recommended: {'yes' if recommended else 'no'}]  yes/no: ").strip().lower()
        answer = reply.startswith("y") if reply else recommended
        who = "human reviewer"
    audit("human", "approval gate", question, f"{'APPROVED' if answer else 'REJECTED'} · {who}")
    return answer


print(f"fast mode: {FAST} · auto-approve: {AUTO_APPROVE}")

In [ ]:
# ── Platform ─────────────────────────────────────────────────────────────────────────────
ORGANIZATION, WORKSPACE = "Lexsi Labs", "TabTunetesting"
PROJECT = "TabTune Demo · Home Credit amount (full flow)"                # created if it does not exist
TASK = "regression"
COMPUTE, UPLOAD_COMPUTE = "T4.small", "small"         # GPU queue for foundation-model builds · CPU size for the upload job
DASHBOARD_POD = "small"                               # compute for drift dashboards and monitors

# ── LLM gateway: role → (provider, model), exactly as registered in the gateway project ──
LLM_PROJECT, LLM_PROJECT_ID = "tabtunedemo", "tabtunedemo_WJQPD747O8"     # text project: display name and project id
LLM_WORKSPACE, LLM_CLIENT_ID = WORKSPACE, "adityatanna"
GATEWAY_MODELS = {
    "agents":      ("Open Router", "anthropic-claude-sonnet-5.5_v1"),     # data-prep agent and retraining agent
    "card_writer": ("Open Router", "anthropic-claude-sonnet-5.5_v1"),     # model-card writer
    # "open_weights": ("Open Router", "<an open-weights model you registered>"),   # add a row to compare it; point "agents" at it to switch
}
AGENT_MAX_TOKENS, CARD_MAX_TOKENS = 4000, 32000       # thinking tokens count against these limits

# ── Demo size and options ────────────────────────────────────────────────────────────────
TARGET_NAME = "credit amount"
N_CONTEXT, N_VALID, N_TEST, N_MONTH, MONTHS = (400, 200, 300, 150, 4) if FAST else (4000, 1000, 3000, 1000, 4)   # a build is capped at 10,000 rows
ROSTER = ["TabICLv2", "XRFM", "Limix", "TabDPT"]                                   # foundation models trained zero-shot on the platform
if FAST:
    ROSTER = ROSTER[:2]
RUN_ENSEMBLE = RUN_LEADERBOARD = RUN_DISTILLATION = True
RUN_SUPPORT_SET = True                                # training-row attribution, where the model and the account allow it
REQUIRE_COMMERCIAL = False                            # True = only models cleared for commercial use may be selected
ENSEMBLE_METRIC, LEADERBOARD_RANK = "rmse", "r2_score"
DISTILLATION_CONFIG = {"student": "lgbm", "gbdt_backend": "lgbm", "temperature": 3.0, "alpha": 0.7, "n_folds": 5, "adaptive_temperature": True, "confidence_weighting": True}
FEATURE_EXCLUDE, LIME_ITERATIONS, TABPFN_TOKEN = [], 200, None
TOP_K_FEATURES = 20 if FAST else 60                   # the preparation agent keeps the strongest features
MAX_RETRAINS = 2                                      # platform training jobs the retraining agent may start in one run
ALERT_EMAILS = []                                     # addresses that receive monitor alerts; empty = no e-mail
FILE_PREFIX, MONITOR_PREFIX = "hc", "HC amount"
POLICY_DECISION = {"decision": "input", "input": "Refer to manual review"}
SEED = 7

In [ ]:
# @title Helpers: idempotent project and upload, training wrapper, model catalogue, evaluation dashboards, safe platform calls { display-mode: "form" }
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from contextlib import contextmanager
from IPython.display import display, Markdown

BLUE, ORANGE, TEAL, GREY, DARK, RED = "#2F5BEA", "#B4541A", "#1B998B", "#9AA0A8", "#16181D", "#D7263D"
pd.set_option("display.max_columns", 50)


def get_or_create(workspace, name, project_type):
    """Reuse a project if it exists, otherwise create it. Safe to rerun."""
    try:
        project = workspace.project(name)
        print(f"reusing project: {name}")
    except Exception:
        project = workspace.create_project(name, modality="tabular", project_type=project_type)
        print(f"created project: {name}")
    return project


def upload_once(project, csv_path, tag, config=None, compute_type=None):
    """Upload a file unless this tag already has it; set the config only on the first upload.

    Workarounds for lexsi-sdk 0.1.69, applied to the first upload only:
      1. upload_data() deletes the freshly uploaded file -> delete_file is suspended during the call.
      2. an empty pred_label is no longer sent -> pred_label="" is put back in the request.
      3. if the platform cannot read the uploaded file back, the SDK fails with "'str' object has no attribute 'get'"
         -> the platform's own message is raised instead.
      4. a file with the same name left by an earlier attempt (or a cancelled upload in the UI) is removed, and the new copy
         is uploaded under a fresh name, so the platform never reads a path that is being deleted.
    """
    import os, shutil
    try:
        files = project.files()
        existing = set(files["file_name"]) if "file_name" in files else set()
    except Exception:
        existing = set()
    name = os.path.basename(csv_path)
    has_config = project.config() != "Not Found"
    if name in existing and (has_config or config is None):
        print(f"already uploaded: {name} ({tag})")
        return
    if config is None or has_config:
        print(project.upload_data(csv_path, tag))
        return

    if name in existing:                                   # left over from a failed first upload or a cancelled UI upload
        print(f"removing the leftover {name} (the project has no configuration yet) →", project.delete_file(name))
        time.sleep(5)
        fresh = csv_path.replace(".csv", f"_{time.strftime('%H%M%S')}.csv")
        shutil.copy(csv_path, fresh)
        csv_path = fresh
        print(f"uploading as {os.path.basename(fresh)}")
    cls, api_cls = type(project), type(project.api_client)
    had_own, real_delete = "delete_file" in cls.__dict__, cls.__dict__.get("delete_file")
    real_post = api_cls.post

    def post_patched(self, uri, payload=None, *a, **k):
        if isinstance(payload, dict) and "unique_identifier" in payload and payload.get("pred_label") is None:
            payload["pred_label"] = ""
        res = real_post(self, uri, payload, *a, **k)
        if "get_Uploaded_file_info" in str(uri) and isinstance(res, dict) and not res.get("success"):
            raise RuntimeError(f"the platform could not read the uploaded file back: {res.get('details') or res.get('result')}. "
                               "If this repeats, set PROJECT to a new name in the config cell and rerun from section 1.")
        return res

    cls.delete_file = lambda self, *a, **k: "skipped"
    api_cls.post = post_patched
    try:
        print(project.upload_data(csv_path, tag, config=config, compute_type=compute_type))
    finally:
        api_cls.post = real_post
        if had_own:
            cls.delete_file = real_delete
        else:
            del cls.delete_file


@contextmanager
def tolerant_drop():
    """project.models() drops columns that some backend builds do not return."""
    orig = pd.DataFrame.drop

    def safe(self, *a, **k):
        k.setdefault("errors", "ignore")
        return orig(self, *a, **k)

    pd.DataFrame.drop = safe
    try:
        yield
    finally:
        pd.DataFrame.drop = orig


RUNS = []          # one record per training call, shown in the run log


def trained_models(project):
    with tolerant_drop():
        df = project.models()
    return df


def summary_of(project, model_name):
    try:
        return project.model_summary(model_name=model_name)
    except Exception as exc:
        print(f"no summary for {model_name}: {exc}")
        return None


def gfi(summary, method):
    """Global feature importance dict {feature: value} for 'shap' or 'lime', or {}."""
    if summary is None:
        return {}
    block = (summary.model_results or {}).get("GFI", {}) or {}
    raw = block.get(f"{method}_gfi") or {}
    out = {}
    for k, v in raw.items():
        try:
            out[k] = float(np.mean(np.abs(list(v.values())))) if isinstance(v, dict) else float(v)
        except Exception:
            pass
    return out


def support_rows(summary):
    if summary is None:
        return pd.DataFrame()
    block = ((summary.model_results or {}).get("GFI", {}) or {}).get("support_set_attribution") or {}
    return pd.DataFrame(block.get("top_support_rows", []))


def top_bar(importance, title, n=15, color=BLUE):
    s = pd.Series(importance).abs().sort_values(ascending=False).head(n)[::-1]
    fig = go.Figure(go.Bar(x=s.values, y=s.index, orientation="h", marker_color=color))
    fig.update_layout(template="plotly_white", height=36 * len(s) + 140, title=title,
                      xaxis_title="mean |attribution|", margin=dict(l=10, r=10, t=60, b=40))
    return fig

def model_names(project):
    try:
        return set(trained_models(project)["model_name"].astype(str))
    except Exception:
        return set()


def train(project, label, **kwargs):
    """train_model with timing, a status line, and the exact name of the model it created."""
    before, t0 = model_names(project), time.time()
    try:
        project.train_model(**kwargs)
        status = "ok"
    except Exception as exc:
        status = f"failed: {str(exc)[:160]}"
        xai = (kwargs.get("data_config") or {}).get("xai_method", [])
        if "support_set" in xai and "not allowed" in str(exc):
            print(f"↺ {label}: support set is not enabled for this account; retrying with SHAP and LIME")
            kwargs["data_config"] = {**kwargs["data_config"], "xai_method": [m for m in xai if m != "support_set"]}
            return train(project, label, **kwargs)
    new = sorted(model_names(project) - before)
    RUNS.append({"run": label, "strategy": kwargs.get("training_strategy"), "model": str(kwargs.get("model")),
                 "created": ", ".join(new), "status": status, "minutes": round((time.time() - t0) / 60, 1)})
    print(("✅ " if status == "ok" else "❌ ") + f"{label} → {', '.join(new) or '-'}: {status} ({RUNS[-1]['minutes']} min)")
    return new[-1] if (status == "ok" and new) else None


def base_data_config(xai):
    return {"tags": ["Training"], "test_tags": ["Test"], "feature_exclude": FEATURE_EXCLUDE,
            "xai_method": xai, "lime_explainability_iterations": LIME_ITERATIONS}


def model_config_for(models):
    models = [models] if isinstance(models, str) else list(models)
    return {"tabpfn_token": TABPFN_TOKEN} if "TabPFNv3" in models and TABPFN_TOKEN else None


def eval_metrics(summary):
    return dict((summary.model_results or {}).get("evalution_metrics", {}) or {}) if summary else {}


def modelling_info(summary):
    return dict((summary.model_results or {}).get("modelling_info", {}) or {}) if summary else {}


def features_used(summary):
    return list((summary.model_results or {}).get("feature_actual_used", []) or []) if summary else []


def ensure_inference(project, model_name):
    """Make `model_name` the active model and enable it for inference."""
    project.activate_model(model_name)
    try:
        project.update_inference_model_status(model_name, True)
    except Exception:
        pass


def case_attributions(case, kind="shap", positive="1"):
    """Signed per-feature attribution dict from a case, picking the positive class when nested."""
    raw = getattr(case, f"{kind}_feature_importance", {}) or {}
    if raw and isinstance(next(iter(raw.values())), dict):
        key = positive if positive in raw else list(raw)[-1]
        raw = raw[key]
    out = {}
    for k, v in raw.items():
        try:
            out[k] = float(v)
        except Exception:
            pass
    return out


def signed_bar(attr, title, n=12):
    s = pd.Series(attr)
    s = s.reindex(s.abs().sort_values(ascending=False).index).head(n)[::-1]
    fig = go.Figure(go.Bar(x=s.values, y=s.index, orientation="h",
                           marker_color=[RED if v > 0 else BLUE for v in s.values]))
    fig.update_layout(template="plotly_white", height=34 * len(s) + 150, title=title,
                      xaxis_title="contribution (red pushes the prediction up)", margin=dict(l=10, r=10, t=60, b=40))
    return fig

from sklearn import metrics as skm


def prediction_columns(result, uploaded):
    """Columns the platform added to the uploaded tag (prediction, probabilities, ...)."""
    return [c for c in result.columns if c not in set(uploaded.columns)]


def positive_proba(result, added, positive="1"):
    """Best guess at the positive-class probability column among the added columns."""
    numeric = [c for c in added if pd.api.types.is_numeric_dtype(result[c])
               and result[c].between(0, 1).all() and result[c].nunique() > 2]
    ranked = sorted(numeric, key=lambda c: (str(positive) not in c.split("_")[-1] and not c.endswith(str(positive)),
                                            "prob" not in c.lower() and "score" not in c.lower()))
    return ranked[0] if ranked else None


def predicted_label(result, added):
    cands = [c for c in added if result[c].nunique() <= 20]
    named = [c for c in cands if "pred" in c.lower()]
    return (named or cands or [None])[0]

def regression_dashboard(y_true, y_pred, title, unit=""):
    """Predicted vs actual, residuals, error by decile and residual distribution."""
    y, f = np.asarray(y_true, float), np.asarray(y_pred, float)
    res = y - f
    out = {"rmse": float(np.sqrt(np.mean(res ** 2))), "mae": float(np.mean(np.abs(res))),
           "r2": float(skm.r2_score(y, f)), "mape_%": float(np.mean(np.abs(res) / np.maximum(np.abs(y), 1e-9)) * 100)}
    dec = pd.qcut(y, 10, labels=False, duplicates="drop")
    by_dec = pd.DataFrame({"dec": dec, "ape": np.abs(res) / np.maximum(np.abs(y), 1e-9) * 100}).groupby("dec")["ape"].median()
    lo, hi = np.percentile(np.r_[y, f], [1, 99])
    fig = make_subplots(rows=2, cols=2, vertical_spacing=0.15,
                        subplot_titles=(f"Predicted vs actual · R² {out['r2']:.3f}", "Residuals vs prediction",
                                        "Median absolute % error by actual-value decile", "Residual distribution"))
    fig.add_trace(go.Scattergl(x=y, y=f, mode="markers", marker=dict(color=BLUE, size=5, opacity=0.45), name="loan"), 1, 1)
    fig.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], line=dict(color=DARK, dash="dot"), name="perfect"), 1, 1)
    fig.add_trace(go.Scattergl(x=f, y=res, mode="markers", marker=dict(color=ORANGE, size=5, opacity=0.45), showlegend=False), 1, 2)
    fig.add_hline(y=0, line_color=DARK, line_dash="dot", row=1, col=2)
    fig.add_trace(go.Bar(x=[f"D{int(d) + 1}" for d in by_dec.index], y=by_dec.values, marker_color=TEAL, showlegend=False), 2, 1)
    fig.add_trace(go.Histogram(x=res, nbinsx=50, marker_color=GREY, showlegend=False), 2, 2)
    fig.update_xaxes(title_text=f"actual {unit}", row=1, col=1); fig.update_yaxes(title_text=f"predicted {unit}", row=1, col=1)
    fig.update_xaxes(title_text=f"predicted {unit}", row=1, col=2); fig.update_yaxes(title_text="% error", row=2, col=1)
    fig.update_layout(template="plotly_white", height=760, title=title, legend=dict(orientation="h", y=-0.08))
    fig.show()
    return out

def preflight_high_cardinality(df, cols, limit=0.95, digits=(4, 3, 2)):
    """The backend drops any column with more than 95% unique values (model.py:494-500), which also
    removes continuous features. Round those columns to fewer significant digits so they survive."""
    report = []
    for c in cols:
        if not pd.api.types.is_numeric_dtype(df[c]):
            continue
        ratio = df[c].nunique() / len(df)
        if ratio <= limit:
            continue
        fixed = None
        for d in digits:
            rounded = df[c].map(lambda v: float(f"{v:.{d}g}") if pd.notna(v) else v)
            if rounded.nunique() / len(df) <= limit:
                fixed = d
                break
        report.append({"feature": c, "unique_ratio": round(ratio, 3),
                       "fix": f"rounded to {fixed} significant digits" if fixed else "dropped by backend"})
        if fixed:
            df[c] = df[c].map(lambda v: float(f"{v:.{fixed}g}") if pd.notna(v) else v)
    return pd.DataFrame(report)

CATALOGUE = pd.DataFrame([
    # model,          classification, regression, support_set, shap_regression, commercial,  note
    ("OrionMoE",      True,  False, False, False, "yes",        "Lexsi mixture-of-experts, inference only"),
    ("OrionMSP",      True,  False, True,  False, "yes",        "Lexsi multi-scale sparse attention"),
    ("OrionBix",      True,  False, True,  False, "yes",        "Lexsi bi-axial attention"),
    ("OrionMSPv1.5",  True,  False, False, False, "yes",        "Lexsi"),
    ("TabICL",        True,  False, True,  False, "yes",        ""),
    ("TabICLv2",      True,  True,  False, True,  "yes",        ""),
    ("TabPFN",        True,  True,  False, True,  "unverified", ""),
    ("TabPFNv26",     True,  True,  True,  True,  "unverified", ""),
    ("TabPFNv3",      True,  True,  True,  True,  "no",         "needs a tabpfn_token"),
    ("TabDPT",        True,  True,  False, True,  "unverified", ""),
    ("Mitra",         True,  True,  False, False, "yes",        "no predict_proba"),
    ("ContextTab",    True,  True,  False, False, "no",         ""),
    ("Limix",         True,  True,  True,  True,  "no",         ""),
    ("TabFM",         True,  True,  False, False, "no",         "Google"),
    ("EXAONETabular", True,  False, False, False, "no",         "LG AI Research"),
    ("ILTM",          True,  True,  False, False, "yes",        ""),
    ("XRFM",          True,  True,  False, True,  "yes",        "trains from scratch"),
], columns=["model", "classification", "regression", "support_set", "shap_regression", "commercial", "note"]).set_index("model")


def xai_for(model, task):
    """Explainability methods the backend accepts for a simple build of `model`."""
    cap = CATALOGUE.loc[model]
    if task == "regression":
        return ["shap", "lime"] if cap.shap_regression else ["lime"]
    methods = ["shap", "lime"]
    if cap.support_set and RUN_SUPPORT_SET:
        methods.append("support_set")
    return methods

def attempt(label, fn, default=None, show=True):
    """Run one platform call; on failure print the platform's message and carry on, so one failing call never stops the demo."""
    t0 = time.time()
    try:
        out = fn()
        if isinstance(out, Exception):                     # some SDK calls return the error instead of raising it
            raise out
        if show:
            print(f"✅ {label} ({time.time() - t0:.1f}s)" + (f" → {out}" if isinstance(out, str) and len(out) < 120 else ""))
        return out
    except Exception as exc:
        print(f"⚠️ {label}: {str(exc)[:300]}")
        return default


def as_frame(obj):
    """Platform list calls return a DataFrame, or a message string when there is nothing yet."""
    return obj if isinstance(obj, pd.DataFrame) else pd.DataFrame()

In [ ]:
organization = lexsi.organization(ORGANIZATION)
workspace = organization.workspace(WORKSPACE)
project = get_or_create(workspace, PROJECT, TASK)          # Organization → Workspace → Project
CATALOGUE.loc[ROSTER]

## 1.2 Endpoint map
Every step in this notebook is one SDK call, and every SDK call is one REST endpoint under `https://api.lexsi.ai`. All of them take the access token; the gateway also accepts an API token in the `x-api-token` header.

In [ ]:
# @title SDK call → REST endpoint { display-mode: "form" }
ENDPOINT_MAP = pd.DataFrame([
    ("Sign in",                    "lexsi.login()",                                         "POST v2/access-token/authorize"),
    ("Project",                    "workspace.create_project(...) · workspace.project(...)", "workspace and project management"),
    ("Upload data + config",       "project.upload_data(file, tag, config=, compute_type=)", "POST v2/project/upload_data"),
    ("Data checks",                "project.data_observations(tag) · data_warnings(tag)",    "POST v2/project/data_summary"),
    ("Train / AutoML",             "project.train_model(model=, training_strategy=, ...)",   "POST v2/users/xai_config_update"),
    ("Model registry",             "project.models() · activate_model(name)",                "v2/ai-models/get_all_models · update_active_model"),
    ("Model summary, global XAI",  "project.model_summary(model_name)",                      "v2/project/get_model_perfermance"),
    ("Batch inference",            "project.model_inference(tag=, model_name=, compute_type=)", "POST v2/ai-models/run_model_on_data"),
    ("Single case + explanations", "project.case_predict(unique_identifier=, xai=, risk_policies=True)", "v2/ai-models/get_case_info_components"),
    ("Observations",               "project.create_observation(...) · observations()",       "POST v2/observations/create_observation"),
    ("Policies",                   "project.create_policy(...) · policies()",                "POST v2/policies/create_policy"),
    ("Drift dashboards",           "project.get_data_drift_dashboard(payload) · get_target_drift_dashboard(payload)", "POST v2/dashboards/generate_dashboard?type=..."),
    ("Monitors",                   "project.create_monitor(payload) · monitors()",           "POST v2/triggers/update_triggers"),
    ("Alerts",                     "project.alerts() · get_alert_details(id)",               "POST v2/triggers/get_executed_triggers"),
    ("LLM gateway",                "text_project.chat_completion(model=, provider=, messages=, session_id=)", "POST gateway/v1/chat/completions"),
    ("Gateway model catalogue",    "text_project.search_text_models(provider) · initialize_text_model(...)", "model search and registration"),
    ("Agent traces",               "text_project.sessions() · messages(session_id) · traces(trace_id)", "GET sessions/get_sessions · sessions/get_session_messages · traces/get_traces"),
], columns=["Step", "SDK call", "REST endpoint"])
display(ENDPOINT_MAP.style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "pre-wrap"})
        .set_caption("Every platform step in this notebook: the SDK call and the endpoint behind it"))

## 1.3 The LLM gateway: one endpoint, many models
`POST /gateway/v1/chat/completions` is a single OpenAI-compatible endpoint in front of every provider registered in a Lexsi text project: OpenAI, Anthropic, Gemini, Groq, Grok, Mistral, Hugging Face, AWS Bedrock, Replicate, Open Router, Together, Lexsi-hosted and self-hosted models. The gateway applies the project's guardrails and stores every call as a **session message with a trace**.

**Are the agents LLMs?** In the open-source notebooks they were rule-based. Here each agent has two parts: deterministic checks that compute the evidence (drift, leakage probes, scores), and an **LLM reviewer** that reads the evidence, decides, and explains its decision in plain English. A human gate then confirms. Policy-mandated steps (known exclusions, constant columns, target leakage) stay with the rules; the LLM only explains them.

**Which model?** Whatever `GATEWAY_MODELS` names for the role. It is set to the model registered in the gateway project today (Claude Sonnet 5.5 through Open Router). To run the agents on an **open-weights** model, register one in the project (cell 1.3c) and point the `agents` role at it; nothing else changes. If the gateway cannot be reached, the agents fall back to their rule defaults and say so.

In [ ]:
# @title Gateway client, agent reviewer and trace log { display-mode: "form" }
import uuid, httpx
try:
    import h2
    HTTP2 = True
except ImportError:
    HTTP2 = False

AGENT_TRACE = []                                   # one row per LLM call made by any agent in this notebook
_SESSION = {"id": str(uuid.uuid4()), "use": True}  # all agent calls share one gateway session, so their traces sit together
LEXSI_BASE = os.environ.get("LEXSI_GATEWAY_BASE", "https://api.lexsi.ai/gateway/v1")


def _secret(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    return os.environ.get(name)


def connect_gateway():
    """Open the Lexsi text project that fronts the LLM gateway. Returns None if it cannot be reached through the SDK."""
    for ws_name in dict.fromkeys([LLM_WORKSPACE, WORKSPACE]):
        try:
            tp = organization.workspace(ws_name).project(LLM_PROJECT)
            print(f"gateway project: {LLM_PROJECT} (workspace {ws_name}) · project id {getattr(tp, 'project_name', LLM_PROJECT_ID)}")
            return tp
        except Exception as exc:
            print(f"text project '{LLM_PROJECT}' not found in workspace '{ws_name}': {str(exc)[:120]}")
    if _secret("LEXSI_API_TOKEN"):
        print("→ using the gateway REST endpoint directly with the LEXSI_API_TOKEN secret")
    else:
        print("→ no gateway available: agents fall back to their rule defaults and the model card to the template writer")
    return None


def gateway_ready():
    return text_project is not None or bool(_secret("LEXSI_API_TOKEN"))


def _chat_sdk(provider, model, messages, max_tokens):
    kw = dict(model=model, messages=messages, provider=provider, max_tokens=max_tokens)
    if _SESSION["use"]:
        try:
            return text_project.chat_completion(session_id=_SESSION["id"], **kw)
        except Exception as exc:
            if "session" not in str(exc).lower():
                raise
            _SESSION["use"] = False                           # the gateway did not accept a client-side session id
    return text_project.chat_completion(**kw)


def _chat_rest(provider, model, messages, max_tokens):
    payload = {"provider": provider, "model": model, "project_name": LLM_PROJECT_ID, "client_id": LLM_CLIENT_ID,
               "max_tokens": max_tokens, "messages": messages, "stream": False}
    post = lambda body: client.post(f"{LEXSI_BASE}/chat/completions", json=body,
                                    headers={"x-api-token": _secret("LEXSI_API_TOKEN"), "Content-Type": "application/json"})
    with httpx.Client(http2=HTTP2, timeout=httpx.Timeout(900, connect=30)) as client:
        r = post({**payload, "session_id": _SESSION["id"]}) if _SESSION["use"] else post(payload)
        if r.status_code in (400, 422) and _SESSION["use"]:
            _SESSION["use"] = False                           # this gateway route does not take a client-side session id
            r = post(payload)
    if r.status_code != 200:
        raise RuntimeError(f"gateway returned {r.status_code}: {r.text[:300]}")
    return r.json()


def _chat(provider, model, messages, max_tokens):
    """One chat completion through the Lexsi gateway: the SDK route first, the REST route (API token) as the fallback."""
    if text_project is not None and _SESSION.get("sdk", True):
        try:
            return _chat_sdk(provider, model, messages, max_tokens), "sdk"
        except Exception as exc:
            if not _secret("LEXSI_API_TOKEN"):
                raise
            _SESSION["sdk"] = False
            print(f"gateway call through the SDK failed ({str(exc)[:160]}) → using the REST endpoint with the API token from here on")
    return _chat_rest(provider, model, messages, max_tokens), "rest"


def llm_call(role, messages, max_tokens=2000, agent=None, step=None):
    """Call the model assigned to `role` in GATEWAY_MODELS and log the call."""
    provider, model = GATEWAY_MODELS[role]
    t0 = time.time()
    body, via = _chat(provider, model, messages, max_tokens)
    if not isinstance(body, dict) or not body.get("choices"):
        raise RuntimeError(f"no answer from the gateway: {str(body)[:300]}")          # e.g. blocked by a guardrail
    ch = body["choices"][0]; msg = ch.get("message") or {}
    text = msg.get("content") or ""
    if isinstance(text, list):
        text = "".join(b.get("text", "") for b in text if isinstance(b, dict))
    usage = body.get("usage") or {}
    out = {"text": text, "finish_reason": str(ch.get("finish_reason")), "seconds": round(time.time() - t0, 1), "provider": provider, "model": model,
           "via": via, "input_tokens": usage.get("prompt_tokens") or 0, "output_tokens": usage.get("completion_tokens") or 0,
           "reasoning": msg.get("reasoning") or msg.get("reasoning_content") or ""}
    AGENT_TRACE.append({"at": pd.Timestamp.now().strftime("%H:%M:%S"), "agent": agent or role, "step": step or "", "provider": provider, "model": model,
                        "via": via, "seconds": out["seconds"], "input_tokens": out["input_tokens"], "output_tokens": out["output_tokens"],
                        "finish_reason": out["finish_reason"], "decision": "", "rationale": ""})
    return out


AGENT_SYSTEM = ("You are the {agent} in a bank's model pipeline. You review ONE proposed action at a time. The evidence was computed by "
                "deterministic checks; do not invent numbers. Reply with JSON only, no other text: "
                '{{"decision": "approve" or "reject", "rationale": "<one or two plain-English sentences that cite the evidence>"}}. '
                "Approve when the evidence supports the action and the action is reversible or reviewed by a human; reject when the "
                "evidence is weak or the action would remove information the model legitimately needs.")


def agent_llm(agent, step, evidence, default="approve", mandatory=False):
    """Ask the agent's LLM for a decision on one step. Rules stay in charge of mandatory steps; the LLM explains them."""
    if not gateway_ready():
        return {"decision": default, "rationale": "rule default (no LLM gateway configured)", "source": "rules"}
    note = " This step is mandated by policy: decide 'approve' and explain why the rule exists." if mandatory else ""
    try:
        res = llm_call("agents", [{"role": "system", "content": AGENT_SYSTEM.format(agent=agent) + note},
                                  {"role": "user", "content": json.dumps({"step": step, **evidence}, default=str)[:6000]}],
                       max_tokens=AGENT_MAX_TOKENS, agent=agent, step=step)
        m = re.search(r"\{.*\}", res["text"], flags=re.S)
        verdict = json.loads(m.group(0)) if m else {}
        decision = str(verdict.get("decision", "")).lower()
        if decision not in ("approve", "reject"):
            raise ValueError(f"unreadable answer: {res['text'][:120]}")
        if mandatory:
            decision = "approve"
        out = {"decision": decision, "rationale": str(verdict.get("rationale", ""))[:400], "source": f"{res['provider']} · {res['model']}"}
        AGENT_TRACE[-1].update(decision=decision, rationale=out["rationale"])
    except Exception as exc:
        out = {"decision": default, "rationale": f"rule default (LLM call failed: {str(exc)[:120]})", "source": "rules"}
    audit(agent, f"{step} · LLM review", out["rationale"], f"{out['decision'].upper()} · {out['source']}")
    return out

def compare_models(question, roles=None, max_tokens=600):
    """Send one prompt to every distinct model configured in GATEWAY_MODELS and compare the answers."""
    if not gateway_ready():
        return pd.DataFrame([{"provider": p, "model": m, "answer": "not called: no gateway configured"} for p, m in dict.fromkeys(GATEWAY_MODELS.values())])
    rows, seen = [], set()
    for role in roles or GATEWAY_MODELS:
        key = GATEWAY_MODELS[role]
        if key in seen:
            continue
        seen.add(key)
        try:
            r = llm_call(role, [{"role": "user", "content": question}], max_tokens=max_tokens, agent="gateway check", step="model comparison")
            rows.append({"provider": key[0], "model": key[1], "seconds": r["seconds"], "input_tokens": r["input_tokens"],
                         "output_tokens": r["output_tokens"], "answer": r["text"].strip()[:300]})
        except Exception as exc:
            rows.append({"provider": key[0], "model": key[1], "answer": f"failed: {str(exc)[:200]}"})
    return pd.DataFrame(rows)

In [ ]:
text_project = connect_gateway()                     # the Lexsi text project that fronts the gateway

In [ ]:
# 1.3b · the same question to every model configured above (add rows to GATEWAY_MODELS to compare more)
compare_models("In one sentence: what does the population stability index (PSI) tell a credit-risk team?")

In [ ]:
# 1.3c · what else the gateway can serve, and how a model is added to the project
REGISTER_MODEL = None        # e.g. ("Open Router", "openai/gpt-oss-120b") to register an open-weights model for the agents
if text_project is not None:
    display(as_frame(attempt("Open Router models matching 'gpt-oss'", lambda: text_project.search_text_models("Open Router", model_name="gpt-oss"))).head(8))
    if REGISTER_MODEL:
        attempt(f"register {REGISTER_MODEL[1]}", lambda: text_project.initialize_text_model(
            model_provider=REGISTER_MODEL[0], model_name=REGISTER_MODEL[1], model_task_type="text-generation", model_architecture="llm"))

# 2 · Product options

## Problem statement
**How much credit should this applicant be offered?** A retail lender recommends a loan amount from the applicant's profile, bureau history and previous applications, to support limit setting and affordability checks.

## Data on the platform
Data is uploaded once per **tag**. Tags select the rows for training, testing, scoring and monitoring.

| Tag | Rows (default) | Used for |
|---|---|---|
| **Training** | 5,000 | the rows foundation models read as context; training data for the other builds |
| **Test** | 3,000 | the platform's evaluation metrics, the final performance report, conformal calibration |
| **ProdM1 … ProdM4** | 4 × 1,000 | four later "months": batch scoring, drift dashboards, monitors, retraining |

## What runs where

| Step | In this notebook | On the Lexsi platform |
|---|---|---|
| Data preparation | profile, agent's checks, feature engineering | data checks on the uploaded tags |
| Agent reasoning | evidence is computed here | the LLM reviewer, through the gateway |
| Training | — | zero-shot foundation models, ensemble, leaderboard, distillation |
| Scoring and explanations | plots | `model_inference`, `case_predict` with SHAP / LIME |
| Guardrails | proposals | observations and policies, applied to every case |
| Monitoring | month-by-month evidence, webhook | drift dashboards, monitors, alerts |
| Documentation | fact sheet, automatic checks | model card written through the gateway, traced |

# 3 · Data preparation

## 3.1 Load

In [ ]:
# @title Home Credit (processed parquet files on Google Drive) { display-mode: "form" }
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass
DATA_DIR = os.environ.get("HC_DATA_DIR", "/content/drive/MyDrive/datasets/home-credit-default-risk/processed")
data = pd.concat([pd.read_parquet(os.path.join(DATA_DIR, f)) for f in ("X_train.parquet", "X_val.parquet")], ignore_index=True)
data.columns = [re.sub(r"[^0-9A-Za-z_]+", "_", str(c)).strip("_") for c in data.columns]
data = data.loc[:, ~data.columns.duplicated()]
UID, TARGET_COL = "SK_ID_CURR", "AMT_CREDIT"
data[TARGET_COL] = pd.to_numeric(data[TARGET_COL], errors="coerce")
data = data[data[TARGET_COL].notna()].reset_index(drop=True)
# Fields that restate the answer or are only known after the loan exists
KNOWN_EXCLUDE = {c: "leaks the target (derived from or equal to the credit amount)" for c in data.columns
                 if c in ("AMT_GOODS_PRICE", "AMT_ANNUITY") or (c.startswith("FE_") and re.search("CREDIT|ANNUITY|GOODS", c))}
if "TARGET" in data:
    KNOWN_EXCLUDE["TARGET"] = "default outcome, observed only after the loan"
# Ratios defined from the credit amount itself (loan length = credit / annuity, payment rate = annuity / credit)
for c in ("FE_LOAN_LENGTH", "FE_PAYMENT_RATE"):
    if c in data:
        KNOWN_EXCLUDE[c] = "computed from the credit amount and the annuity"
CLASS_NAMES = None
GROUP_PREFIX, GROUP_LABEL = "CODE_GENDER", "gender"
print(f"{len(data):,} applications · {data.shape[1] - 2} candidate features · median credit {data[TARGET_COL].median():,.0f}")
display(data.sample(5, random_state=1).iloc[:, :12])

In [ ]:
# @title Four disjoint sets: context, validation, test and a production stream { display-mode: "form" }
from sklearn.model_selection import train_test_split
rng = np.random.default_rng(SEED)
need = N_CONTEXT + N_VALID + N_TEST + N_MONTH * MONTHS
work = data.sample(min(len(data), max(need * 3, need)), random_state=SEED).reset_index(drop=True)
strat = work[TARGET_COL] if TASK == "classification" else pd.qcut(work[TARGET_COL].rank(method="first"), 10, labels=False)
dev, stream = train_test_split(work, train_size=N_CONTEXT + N_VALID + N_TEST, stratify=strat, random_state=SEED)
s2 = strat.loc[dev.index]
ctx, rest = train_test_split(dev, train_size=N_CONTEXT, stratify=s2, random_state=SEED)
val, test = train_test_split(rest, train_size=N_VALID, stratify=s2.loc[rest.index], random_state=SEED)
ctx, val, test, stream = (d.reset_index(drop=True) for d in (ctx, val, test, stream))
ALL_FEATURES = [c for c in data.columns if c not in (UID, TARGET_COL)]
sizes = pd.Series({"context (training)": len(ctx), "validation": len(val), "test": len(test), "production stream": len(stream)})
fig = make_subplots(rows=1, cols=2, column_widths=[0.45, 0.55], subplot_titles=("Rows per set", f"Target in each set ({TARGET_NAME})"))
fig.add_trace(go.Bar(x=sizes.index, y=sizes.values, marker_color=[BLUE, TEAL, ORANGE, GREY], text=sizes.values, textposition="outside", showlegend=False), 1, 1)
for name, d, col in (("context", ctx, BLUE), ("validation", val, TEAL), ("test", test, ORANGE)):
    if TASK == "classification":
        fig.add_trace(go.Bar(x=[name], y=[d[TARGET_COL].mean() * 100], marker_color=col, showlegend=False, text=[f"{d[TARGET_COL].mean():.1%}"], textposition="outside"), 1, 2)
    else:
        fig.add_trace(go.Violin(y=d[TARGET_COL], name=name, line_color=col, box_visible=True, showlegend=False), 1, 2)
fig.update_layout(template="plotly_white", height=420, title="Data split · stratified so every set has the same target mix")
fig.show()
record("data", source="Lending Club" if TASK == "classification" else "Home Credit", rows_available=len(data), context_rows=len(ctx),
       validation_rows=len(val), test_rows=len(test), candidate_features=len(ALL_FEATURES), target=TARGET_COL)

## 3.2 Data quality and distributions

In [ ]:
# @title Profile dashboard: types, missing values, cardinality, skew { display-mode: "form" }
NUM = [c for c in ALL_FEATURES if pd.api.types.is_numeric_dtype(ctx[c])]
CAT = [c for c in ALL_FEATURES if c not in NUM]
miss = ctx[ALL_FEATURES].isna().mean().mul(100)
uniq = ctx[ALL_FEATURES].nunique()
const = [c for c in ALL_FEATURES if uniq[c] <= 1]
binary = [c for c in NUM if ctx[c].dropna().isin([0, 1]).all() and c not in const]
cont = [c for c in NUM if c not in binary and uniq[c] > 20]
skew = ctx[cont].skew().abs().sort_values(ascending=False) if cont else pd.Series(dtype=float)

kinds = pd.Series({"continuous": len(cont), "binary / one-hot": len(binary), "low-cardinality numeric": len(NUM) - len(cont) - len(binary), "categorical": len(CAT)})
fig = make_subplots(rows=2, cols=3, vertical_spacing=0.2, specs=[[{"type": "domain"}, {}, {}], [{}, {}, {}]],
                    subplot_titles=("Feature types", "Most incomplete features (% missing)", "Distinct values per feature (log)",
                                    "Most skewed features", "Heaviest-tailed feature · raw", "Same feature · signed log"))
fig.add_trace(go.Pie(labels=kinds.index, values=kinds.values, hole=0.55, marker_colors=[BLUE, TEAL, GOLD, ORANGE], showlegend=False, textinfo="label+value"), 1, 1)
tm = miss.sort_values(ascending=False).head(12)[::-1]
fig.add_trace(go.Bar(x=tm.values, y=[c[:26] for c in tm.index], orientation="h", marker_color=[RED if v > 50 else GREY for v in tm.values], showlegend=False), 1, 2)
fig.add_trace(go.Histogram(x=np.log10(uniq.clip(lower=1)), nbinsx=30, marker_color=BLUE, showlegend=False), 1, 3)
if len(skew):
    sk = skew.head(10)[::-1]; f0 = skew.index[0]
    fig.add_trace(go.Bar(x=sk.values, y=[c[:26] for c in sk.index], orientation="h", marker_color=ORANGE, showlegend=False), 2, 1)
    fig.add_trace(go.Histogram(x=ctx[f0], nbinsx=60, marker_color=GREY, showlegend=False), 2, 2)
    fig.add_trace(go.Histogram(x=np.sign(ctx[f0]) * np.log1p(ctx[f0].abs()), nbinsx=60, marker_color=TEAL, showlegend=False), 2, 3)
fig.update_layout(template="plotly_white", height=720, title="Data profile of the context set")
fig.show()
record("data", features_with_missing=int((miss > 0).sum()), constant_features=len(const), categorical_features=len(CAT),
       heavily_skewed_features=int((skew > 2).sum()))

In [ ]:
# @title Signal dashboard: which features move the target, and which repeat each other { display-mode: "form" }
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression

def encode(df, cols=None):
    """Numeric view of a frame: categoricals become stable integer codes, missing values stay missing."""
    cols = cols or ALL_FEATURES
    out = df[cols].copy()
    for c in cols:
        if c in CAT_CODES:
            out[c] = out[c].map(CAT_CODES[c]).astype(float)
    return out.replace([np.inf, -np.inf], np.nan)

from sklearn import metrics as skm
PRIMARY = "roc_auc" if TASK == "classification" else "r2"

def score_of(model, X):
    """Probability of the positive class, or the predicted value."""
    if TASK == "classification":
        p = np.asarray(model.predict_proba(X))
        return p[:, 1] if p.ndim == 2 else p
    return np.asarray(model.predict(X), dtype=float)

def metric(y, s):
    return float(skm.roc_auc_score(y, s) if TASK == "classification" else skm.r2_score(y, s))

CAT_CODES = {c: {v: i for i, v in enumerate(sorted(data[c].dropna().astype(str).unique()))} for c in CAT}
for c in CAT:
    for d in (ctx, val, test, stream):
        d[c] = d[c].astype(str).where(d[c].notna(), None)
Xc_num = encode(ctx)
fill = Xc_num.median()
mi_fn = mutual_info_classif if TASK == "classification" else mutual_info_regression
MI = pd.Series(mi_fn(Xc_num.fillna(fill).fillna(0), ctx[TARGET_COL], random_state=0), index=ALL_FEATURES).sort_values(ascending=False)
top = MI.head(16).index.tolist()
corr = Xc_num[top].corr()
tcorr = Xc_num.corrwith(ctx[TARGET_COL].astype(float)).abs()

fig = make_subplots(rows=1, cols=2, column_widths=[0.38, 0.62], subplot_titles=("Mutual information with the target (top 16)", "Correlation among those features"))
fig.add_trace(go.Bar(x=MI.head(16).values[::-1], y=[c[:28] for c in top[::-1]], orientation="h", marker_color=BLUE, showlegend=False), 1, 1)
fig.add_trace(go.Heatmap(z=corr.values, x=[c[:16] for c in top], y=[c[:16] for c in top], colorscale="RdBu_r", zmin=-1, zmax=1), 1, 2)
fig.update_layout(template="plotly_white", height=600, title="Signal and redundancy")
fig.show()

fig = make_subplots(rows=1, cols=4, subplot_titles=[c[:26] for c in top[:4]])
for j, c in enumerate(top[:4], 1):
    b = pd.qcut(Xc_num[c].rank(method="first"), 10, labels=False)
    g = pd.DataFrame({"b": b, "x": Xc_num[c], "y": ctx[TARGET_COL]}).groupby("b").agg(x=("x", "median"), y=("y", "mean"))
    fig.add_trace(go.Scatter(x=g.x, y=g.y * (100 if TASK == "classification" else 1), mode="lines+markers", line=dict(color=ORANGE, width=3), showlegend=False), 1, j)
fig.update_yaxes(title_text="default rate (%)" if TASK == "classification" else f"mean {TARGET_COL}", row=1, col=1)
fig.update_layout(template="plotly_white", height=360, title="How the target responds across deciles of the four strongest features")
fig.show()

## 3.3 Data preparation agent, reviewed by an LLM
The agent works through the checks a careful analyst would make. For each one it **inspects** the data, **proposes** an action with its evidence, sends the proposal to its **LLM reviewer** through the Lexsi gateway, waits at a **human approval gate** (pre-filled with the reviewer's decision), **applies** what was approved, and **logs** it.

- The checks are deterministic: missing values, constants, identifiers, leakage, redundancy, signal.
- **Too good to be true:** a quick probe model is fitted on the remaining features. If it scores above a ceiling no real lending model reaches, something is restating the answer; the agent finds the feature the probe leans on and proposes removing it.
- The LLM sees only aggregate evidence (feature names, counts, scores), never customer rows. Its rationale is stored in the audit trail and in the agent trace (section 10).

In [ ]:
# @title The data preparation agent (definition) { display-mode: "form" }
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split as _tts

LEAK_CEILING = 0.97 if TASK == "classification" else 0.85


class DataPrepAgent:
    def __init__(self, frame, features):
        self.frame, self.features, self.steps, self.dropped, self.probe_log = frame, list(features), [("candidate features", len(features))], {}, []

    def _propose(self, name, cols, reason, recommended=True):
        cols = [c for c in cols if c in self.features]
        if not cols:
            audit("prep-agent", name, "nothing to do")
            return
        audit("prep-agent", name, f"{len(cols)} feature(s): {', '.join(cols[:5])}{'…' if len(cols) > 5 else ''} — {reason}")
        if human_gate(f"{name}: remove {len(cols)} feature(s)?", recommended):
            self.features = [c for c in self.features if c not in cols]
            self.dropped.update({c: name for c in cols})
        self.steps.append((name, len(self.features)))

    def _probe(self, importance=True):
        """Fit a quick gradient-boosting probe on the current features; return its holdout score and the feature it relies on most."""
        X = encode(self.frame, self.features); y = self.frame[TARGET_COL]
        n = min(len(X), 4000)
        X, y = X.iloc[:n], y.iloc[:n]
        strat = y if TASK == "classification" else None
        Xa, Xb, ya, yb = _tts(X, y, test_size=0.3, random_state=0, stratify=strat)
        Probe_ = HistGradientBoostingClassifier if TASK == "classification" else HistGradientBoostingRegressor
        m = Probe_(max_iter=200, random_state=0).fit(Xa, ya)
        s = metric(yb, m.predict_proba(Xb)[:, 1] if TASK == "classification" else m.predict(Xb))
        if not importance:
            return s, None
        imp = permutation_importance(m, Xb, yb, n_repeats=2, random_state=0, scoring="roc_auc" if TASK == "classification" else "r2")
        imp = pd.Series(imp.importances_mean, index=Xb.columns).sort_values(ascending=False)
        return s, imp

    def _drop_score(self, feature):
        keep = self.features
        self.features = [f for f in keep if f != feature]
        try:
            return self._probe(importance=False)[0]
        finally:
            self.features = keep

    def leakage_probe(self, max_rounds=8):
        for r in range(max_rounds):
            score, imp = self._probe()
            if score > LEAK_CEILING:
                # leaks often need two fields together, so test removing each top suspect; prefer an engineered field whose removal
                # does real damage (a raw application field such as income is rarely the culprit on its own)
                suspects = imp.head(4).index.tolist()
                after = {f: self._drop_score(f) for f in suspects}
                best_drop = score - min(after.values())
                engineered = [f for f in sorted(suspects, key=after.get) if f.lower().startswith("fe_") and score - after[f] >= 0.25 * best_drop]
                top = engineered[0] if engineered else min(suspects, key=after.get)
            else:
                top = imp.index[0]
            share = float(imp[top] / max(imp.clip(lower=0).sum(), 1e-9))
            self.probe_log.append({"round": r + 1, "probe_score": score, "top_feature": top, "importance_share": share})
            if score <= LEAK_CEILING:
                audit("prep-agent", "too good to be true", f"probe {PRIMARY} {score:.3f} ≤ ceiling {LEAK_CEILING}", "believable: stop")
                break
            n_before = len(self.features)
            self._propose("too good to be true", [top], f"probe {PRIMARY} {score:.3f} is above {LEAK_CEILING}; the probe leans on this feature "
                          f"for {share:.0%} of its accuracy, and removing it breaks the probe's shortcut, so it probably restates the answer")
            if len(self.features) == n_before:
                break
        return self

    def run(self):
        f = self.frame
        self._propose("known exclusions", list(KNOWN_EXCLUDE), "identifiers or fields flagged by the data owner")
        self._propose("constant columns", const, "one value only, no information")
        self._propose("mostly missing", miss[miss > 80].index.tolist(), "more than 80% missing")
        ident = [c for c in self.features if c in CAT and uniq[c] > 0.5 * len(f)]
        self._propose("identifier-like text", ident, "almost one distinct value per row")
        leak = tcorr[(tcorr > 0.95)].index.tolist() if TASK == "regression" else tcorr[tcorr > 0.98].index.tolist()
        self._propose("target leakage", leak, "near-perfect correlation with the target; it would not exist at decision time")
        self.leakage_probe()
        strong = [c for c in MI.index if c in self.features and c in NUM][:150]
        cm = Xc_num[strong].corr().abs()
        twins = []
        for i, a in enumerate(strong):
            if a in twins:
                continue
            twins += [b for b in strong[i + 1:] if b not in twins and cm.loc[a, b] > 0.97]
        self._propose("redundant twins", twins, "correlated above 0.97 with a stronger feature")
        weak = [c for c in MI.index if c in self.features][TOP_K_FEATURES:]
        self._propose(f"keep the {TOP_K_FEATURES} strongest", weak, "little or no signal; fewer features means faster scoring and clearer explanations")
        return self

In [ ]:
# @title LLM review step for the agent { display-mode: "form" }
MANDATORY_STEPS = {"known exclusions", "constant columns", "target leakage"}      # rules decide; the LLM explains


class LLMDataPrepAgent(DataPrepAgent):
    """Same checks as the rule-based agent. Each proposal is reviewed by an LLM through the Lexsi gateway before the human gate."""

    def _propose(self, name, cols, reason, recommended=True):
        live = [c for c in cols if c in self.features]
        if live:
            evidence = {"proposal": f"remove {len(live)} feature(s) from the model's inputs", "features": live[:20], "reason": reason,
                        "task": TASK, "target": TARGET_COL, "features_remaining_before": len(self.features),
                        "signal_rank_of_features": {c: int(MI.index.get_loc(c)) + 1 for c in live[:10] if c in MI.index}}
            verdict = agent_llm("data-prep agent", name, evidence, default="approve" if recommended else "reject",
                                mandatory=name.split(":")[0] in MANDATORY_STEPS)
            recommended = verdict["decision"] == "approve"
            reason = f"{reason} · reviewer: {verdict['rationale']}"
        super()._propose(name, cols, reason, recommended)

In [ ]:
prep_agent = LLMDataPrepAgent(ctx, ALL_FEATURES)
prep_agent.run()
FEATURES = prep_agent.features

In [ ]:
# @title What the agent did { display-mode: "form" }
steps = pd.DataFrame(prep_agent.steps, columns=["step", "features left"])
why = pd.Series(prep_agent.dropped).value_counts()
probe = pd.DataFrame(prep_agent.probe_log)
fig = make_subplots(rows=1, cols=3, column_widths=[0.42, 0.28, 0.3], specs=[[{}, {"type": "domain"}, {}]],
                    subplot_titles=("Features remaining after each step", "Why features were removed", f"'Too good to be true' probe ({PRIMARY})"))
fig.add_trace(go.Funnel(y=steps.step, x=steps["features left"], marker_color=BLUE, textinfo="value", showlegend=False), 1, 1)
fig.add_trace(go.Pie(labels=why.index, values=why.values, hole=0.5, marker_colors=PALETTE, showlegend=False, textinfo="label+value"), 1, 2)
if len(probe):
    fig.add_trace(go.Scatter(x=probe["round"], y=probe.probe_score, mode="lines+markers+text", text=probe.top_feature.str[:18], textposition="top center",
                             line=dict(color=ORANGE, width=3), showlegend=False), 1, 3)
    fig.add_trace(go.Scatter(x=[probe["round"].min() - 0.3, probe["round"].max() + 0.3], y=[LEAK_CEILING] * 2, mode="lines",
                             line=dict(color=RED, dash="dot"), showlegend=False, hoverinfo="skip"), 1, 3)
    fig.update_xaxes(title_text="round (label = feature the probe relied on most)", dtick=1, row=1, col=3)
fig.update_layout(template="plotly_white", height=520, title=f"Data preparation agent · {len(ALL_FEATURES)} → {len(FEATURES)} features")
fig.show()
leak_steps = ("known exclusions", "target leakage", "too good to be true")
record("preparation", features_before=len(ALL_FEATURES), features_after_agent=len(FEATURES), features_removed=len(prep_agent.dropped),
       leakage_features_removed=int(pd.Series(prep_agent.dropped).isin(leak_steps).sum()),
       probe_score_first=float(probe.probe_score.iloc[0]) if len(probe) else float("nan"),
       probe_score_final=float(probe.probe_score.iloc[-1]) if len(probe) else float("nan"), leak_ceiling=LEAK_CEILING)

In [ ]:
pd.DataFrame([t for t in AGENT_TRACE if t["agent"] == "data-prep agent"], columns=["step", "model", "seconds", "decision", "rationale"])   # the reviewer's decisions

## 3.4 Automated feature engineering

In [ ]:
# @title Generate candidate features, keep the ones a probe model says help { display-mode: "form" }
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.model_selection import cross_val_score

base = [c for c in MI.index if c in FEATURES and c in cont][:8]
recipes = []
for i, a in enumerate(base):
    for b in base[i + 1:]:
        recipes += [(f"fe_prod__{a}__{b}", "prod", a, b), (f"fe_diff__{a}__{b}", "diff", a, b)]
        if (ctx[b] > 0).all():
            recipes.append((f"fe_ratio__{a}__{b}", "ratio", a, b))

def apply_recipes(df, recs):
    out = {}
    for name, op, a, b in recs:
        out[name] = df[a] * df[b] if op == "prod" else df[a] - df[b] if op == "diff" else df[a] / df[b].where(df[b] > 0)
    return pd.DataFrame(out, index=df.index).replace([np.inf, -np.inf], np.nan)

cand = apply_recipes(ctx, recipes)
cand_mi = pd.Series(mi_fn(cand.fillna(cand.median()).fillna(0), ctx[TARGET_COL], random_state=0), index=cand.columns) if len(recipes) else pd.Series(dtype=float)
parent = pd.Series({n: max(MI[a], MI[b]) for n, _, a, b in recipes})
gain = (cand_mi - parent).sort_values(ascending=False)
KEPT = gain.head(8).index.tolist()
Probe = HistGradientBoostingRegressor if TASK == "regression" else HistGradientBoostingClassifier
scoring = "r2" if TASK == "regression" else "roc_auc"
cv = lambda X: cross_val_score(Probe(max_iter=120, random_state=0), X, ctx[TARGET_COL], cv=3, scoring=scoring).mean()
base_X = encode(ctx, FEATURES)
score_raw, score_fe = cv(base_X), (cv(pd.concat([base_X, cand[KEPT]], axis=1)) if KEPT else np.nan)
helps = bool(KEPT) and score_fe > score_raw + 0.0005
audit("prep-agent", "feature engineering", f"probe {scoring} {score_raw:.4f} → {score_fe:.4f} with {len(KEPT)} engineered features",
      "worth adding" if helps else "no gain: keep raw features")
ADD_ENGINEERED = helps and human_gate(f"Add {len(KEPT)} engineered features?", True)
KEPT_RECIPES = [r for r in recipes if r[0] in KEPT] if ADD_ENGINEERED else []
if ADD_ENGINEERED:
    for d in (ctx, val, test, stream):
        fe = apply_recipes(d, KEPT_RECIPES)
        for c in fe.columns:
            d[c] = fe[c]
    FEATURES = FEATURES + KEPT
    MI = pd.concat([MI, cand_mi[KEPT]]).sort_values(ascending=False)

fig = make_subplots(rows=1, cols=3, column_widths=[0.36, 0.4, 0.24],
                    subplot_titles=(f"{len(recipes)} candidates: signal vs best parent", "Best candidates: gain over parents", f"Probe model ({scoring}, 3-fold)"))
if len(recipes):
    fig.add_trace(go.Scatter(x=parent, y=cand_mi, mode="markers", showlegend=False,
                             marker=dict(color=[ORANGE if n in KEPT else GREY for n in cand_mi.index], size=[10 if n in KEPT else 5 for n in cand_mi.index])), 1, 1)
    g = gain[KEPT][::-1]
    fig.add_trace(go.Bar(x=g.values, y=[n.replace("fe_", "").replace("__", " · ")[:40] for n in g.index], orientation="h", marker_color=ORANGE, showlegend=False), 1, 2)
fig.add_trace(go.Bar(x=["raw", "+ engineered"], y=[score_raw, score_fe], marker_color=[GREY, ORANGE], text=[f"{score_raw:.4f}", f"{score_fe:.4f}"], textposition="outside", showlegend=False), 1, 3)
fig.update_yaxes(range=[np.nanmin([score_raw, score_fe]) * 0.97, np.nanmax([score_raw, score_fe]) * 1.02], row=1, col=3)
fig.update_layout(template="plotly_white", height=520, title="Automated feature engineering · " + ("added to the feature set" if ADD_ENGINEERED else "not added: no measurable gain"))
fig.show()
record("features", final_feature_count=len(FEATURES), engineered_features_added=len(KEPT_RECIPES), strongest_features=", ".join([c for c in MI.index if c in FEATURES][:5]))

In [ ]:
# @title The prepared data in two dimensions, and the model-ready frames { display-mode: "form" }
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
Xtr, ytr = ctx[FEATURES], ctx[TARGET_COL]
Xva, yva = val[FEATURES], val[TARGET_COL]
Xte, yte = test[FEATURES], test[TARGET_COL]
E = encode(ctx, FEATURES); FILL = E.median()
Z = PCA(n_components=2, random_state=0).fit_transform(StandardScaler().fit_transform(E.fillna(FILL).fillna(0)))
fig = go.Figure()
if TASK == "classification":
    for cls, col in ((0, BLUE), (1, RED)):
        m = (ytr == cls).to_numpy()
        fig.add_trace(go.Scattergl(x=Z[m, 0], y=Z[m, 1], mode="markers", name=CLASS_NAMES[cls], marker=dict(size=4, color=col, opacity=0.45)))
else:
    fig.add_trace(go.Scattergl(x=Z[:, 0], y=Z[:, 1], mode="markers", marker=dict(size=4, color=np.log1p(ytr), colorscale="Viridis", opacity=0.6, colorbar=dict(title="log target"))))
fig.update_layout(template="plotly_white", height=480, title="Context set after preparation (PCA projection)")
fig.show()
display(Markdown(f"""
### Data preparation: conclusions
- **{len(ALL_FEATURES)} candidate features became {len(FEATURES)}** after the agent's approved steps; {FACTS['preparation']['leakage_features_removed']} were removed as leakage or known exclusions.
- **{FACTS['data']['features_with_missing']} features have missing values** and {FACTS['data']['heavily_skewed_features']} are heavily skewed; the foundation models apply their own robust scaling and missing-value handling, so no manual imputation was needed.
- **Engineered features were {'added' if ADD_ENGINEERED else 'not added'}** (probe {scoring} {score_raw:.4f} → {score_fe:.4f}).
- Every step is in the audit trail at the end of the notebook.
"""))

## 3.5 Upload to the platform
One file per tag: the unique identifier, the features the agent kept, and the label. The first upload carries the project configuration. A pre-flight check rounds continuous features that the platform would otherwise drop as identifier-like (more than 95% unique values).

In [ ]:
# @title Production months, pre-flight check and one CSV per tag { display-mode: "form" }
from sklearn.model_selection import train_test_split

def psi(ref, new, bins=10):
    ref, new = pd.Series(ref).dropna(), pd.Series(new).dropna()
    if not len(ref) or not len(new):
        return 0.0
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return 0.0
    edges[0], edges[-1] = -np.inf, np.inf
    a = np.histogram(ref, edges)[0] / len(ref) + 1e-4; b = np.histogram(new, edges)[0] / len(new) + 1e-4
    return float(np.sum((b - a) * np.log(b / a)))

# Production months: real rows, resampled so the applicant mix shifts a little more each month on the strongest feature (month 1 is unshifted)
cont_kept = [c for c in MI.index if c in FEATURES and c in cont]
drift_feature = cont_kept[0]
z = ((stream[drift_feature] - stream[drift_feature].mean()) / (stream[drift_feature].std() + 1e-9)).fillna(0).clip(-3, 3)
pool, months = stream.copy(), []
for m in range(MONTHS):
    pick = pool.sample(min(N_MONTH, len(pool)), weights=np.exp(0.35 * m * z.loc[pool.index]), random_state=m)
    pool = pool.drop(pick.index); months.append(pick.reset_index(drop=True))
MONTH_TAGS = [f"ProdM{m + 1}" for m in range(MONTHS)]

# One file per tag: the unique id, the features the agent kept, and the label
COLS = [UID] + FEATURES + [TARGET_COL]
frames = {"Training": pd.concat([ctx, val], ignore_index=True)[COLS], "Test": test[COLS].copy(), **{t: mo[COLS].copy() for t, mo in zip(MONTH_TAGS, months)}}
both = pd.concat(frames.values(), keys=frames.keys())
fixes = preflight_high_cardinality(both, [c for c in FEATURES if c in NUM or c.startswith("fe_")])
if TASK == "classification":
    both[TARGET_COL] = both[TARGET_COL].astype(int).astype(str)
frames = {t: both.loc[t].reset_index(drop=True) for t in frames}
DATA_OUT = "/content" if os.path.isdir("/content") else "."
PATHS = {t: f"{DATA_OUT}/{FILE_PREFIX}_{t.lower()}.csv" for t in frames}
for t, d in frames.items():
    d.to_csv(PATHS[t], index=False)
display(fixes if len(fixes) else Markdown("No feature would be dropped by the platform's 95%-unique rule."))
display(pd.DataFrame({"rows": {t: len(d) for t, d in frames.items()}, "file": PATHS}).rename_axis("tag"))
record("data", platform_tags=", ".join(frames), training_rows_uploaded=len(frames["Training"]), test_rows_uploaded=len(frames["Test"]))

In [ ]:
config = {"unique_identifier": UID, "true_label": TARGET_COL, "pred_label": "", "feature_exclude": [],
          "drop_duplicate_uid": True, "handle_errors": True}
upload_once(project, PATHS["Training"], "Training", config=config, compute_type=UPLOAD_COMPUTE)     # first upload sets the project config
upload_once(project, PATHS["Test"], "Test")
for tag in MONTH_TAGS:
    upload_once(project, PATHS[tag], tag)                                                         # production months, for drift monitoring
project.files()

**Data checks on the platform.** `data_observations` and `data_warnings` report what the platform found in the uploaded tag.

In [ ]:
data_obs = attempt("data observations on the Training tag", lambda: project.data_observations("Training"))
data_warn = attempt("data warnings on the Training tag", lambda: project.data_warnings("Training"))
display(as_frame(data_obs).head(10)); display(as_frame(data_warn).head(10))

# 4 · Model training on the platform

## 4.1 Zero-shot foundation models
One `project.train_model(...)` call per model. `tunning_strategy="inference"` means zero-shot: the model reads the Training tag as context and is evaluated on the Test tag, with no gradient training. Each build is stored in the project's model registry with its metrics and explanations.

In [ ]:
BUILDS = {}
for model in ROSTER:                                                    # one zero-shot build per foundation model
    BUILDS[model] = train(project, f"{model} · zero-shot", model=model, training_strategy="simple", tunning_strategy="inference",
                          compute_type=COMPUTE, data_config=base_data_config(xai_for(model, TASK)), model_config=model_config_for(model))
BUILDS = {m: n for m, n in BUILDS.items() if n}

## 4.2 Ensemble, leaderboard and distillation
Three more training strategies on the same call:

- **ensemble**: greedy selection over the foundation models;
- **leaderboard**: the platform's AutoML race, which trains the candidates and keeps the winner;
- **distillation**: a fast tree student learns from the foundation-model teachers, for low-latency scoring.

In [ ]:
pool = list(BUILDS)[:3]
ADV = {}
if RUN_ENSEMBLE and len(pool) >= 2:
    ADV["Ensemble"] = train(project, "Ensemble · greedy selection", model=pool, training_strategy="ensemble", tunning_strategy="inference",
                            compute_type=COMPUTE, data_config=base_data_config(["lime"]), model_config=model_config_for(pool),
                            ensemble_config={"ensemble_strategy": "greedy_selection", "metric": ENSEMBLE_METRIC, "weight_scheme": "performance",
                                             "cv_folds": 5, "holdout_fraction": 0.2, "greedy_ensemble_size": 50})
if RUN_LEADERBOARD and len(BUILDS) >= 2:
    ADV["Leaderboard"] = train(project, "Leaderboard (platform AutoML race)", model=list(BUILDS), training_strategy="leaderboard",
                               tunning_strategy="inference", compute_type=COMPUTE, data_config=base_data_config(["shap", "lime"] if TASK == "classification" else ["lime"]),
                               model_config=model_config_for(list(BUILDS)), leaderboard_config={"rank_by": LEADERBOARD_RANK})
if RUN_DISTILLATION and len(pool) >= 2:
    ADV["Distilled student"] = train(project, "Distillation to a fast student", model=pool if TASK == "classification" else pool[0],
                                     training_strategy="distillation", compute_type=COMPUTE, data_config=base_data_config(["shap", "lime"]),
                                     model_config=model_config_for(pool), distillation_config=DISTILLATION_CONFIG)
ADV = {k: v for k, v in ADV.items() if v}

## 4.3 Model selection
**Rule:** the highest platform score on the Test tag wins. `REQUIRE_COMMERCIAL = True` restricts the choice to models whose weights are cleared for commercial use. The selected build is activated in the registry.

In [ ]:
# @title Leaderboard from the platform's metrics, and the selection { display-mode: "form" }
METRIC_KEY = "RocAucScore" if TASK == "classification" else "r2_score"
SHOW = ["RocAucScore", "F1Score", "Precision", "Recall", "Accuracy"] if TASK == "classification" else ["r2_score", "RootMeanSquaredError", "Mean_Absolute_Error", "Mean_Absolute_Per_Error"]
ALL_BUILDS = {**BUILDS, **ADV}
SUMMARIES = {label: summary_of(project, name) for label, name in ALL_BUILDS.items()}
rows = []
for label, name in ALL_BUILDS.items():
    em = eval_metrics(SUMMARIES[label])
    rows.append({"build": label, "platform model": name, "kind": "zero-shot foundation model" if label in BUILDS else "advanced build",
                 **{k: em.get(k) for k in SHOW}, "commercial": CATALOGUE.loc[label, "commercial"] if label in CATALOGUE.index else "-"})
assert rows, "No platform build succeeded: read the ❌ messages in section 4.1 (compute type, model access or data config)."
board = pd.DataFrame(rows).set_index("build")
board[METRIC_KEY] = pd.to_numeric(board[METRIC_KEY], errors="coerce")
board = board.sort_values(METRIC_KEY, ascending=False)
eligible = board[board.commercial != "no"] if REQUIRE_COMMERCIAL else board
CHAMPION = eligible[METRIC_KEY].idxmax()
CHAMPION_BUILD = ALL_BUILDS[CHAMPION]
fig = go.Figure(go.Bar(x=board[METRIC_KEY], y=board.index, orientation="h", text=board[METRIC_KEY].round(3), textposition="outside",
                       marker_color=[BLUE if b == CHAMPION else (TEAL if board.loc[b, "kind"].startswith("advanced") else GREY) for b in board.index]))
fig.update_layout(template="plotly_white", height=90 + 46 * len(board), yaxis=dict(autorange="reversed"),
                  title=f"Platform leaderboard on the Test tag · {METRIC_KEY} · selected: {CHAMPION} ({CHAMPION_BUILD})")
fig.show()
display(board.style.format(precision=3, na_rep="-"))
audit("automl", "model selection", f"{len(board)} platform builds scored on the Test tag; best {CHAMPION} at {board.loc[CHAMPION, METRIC_KEY]:.4f}", f"SELECTED {CHAMPION_BUILD}")
record("selection", candidates_raced=len(ROSTER) + len(ADV), candidates_succeeded=len(board), selected_model=CHAMPION, platform_model_name=CHAMPION_BUILD,
       selection_metric=METRIC_KEY, selected_test_tag_score=float(board.loc[CHAMPION, METRIC_KEY]),
       selection_rule="highest platform score on the Test tag" + ("; commercial-use weights only" if REQUIRE_COMMERCIAL else ""))
record("training", platform="Lexsi", adaptation="zero-shot (tunning_strategy='inference')" if CHAMPION in BUILDS else "advanced platform build",
       compute=COMPUTE, training_rows=len(frames["Training"]), features=len(FEATURES))

In [ ]:
ensure_inference(project, CHAMPION_BUILD)                    # activate the selected build and enable it for inference
champion_summary = project.model_summary(model_name=CHAMPION_BUILD)

## 4.4 Global explainability
SHAP and LIME were computed by the platform during training and are stored with the build.

In [ ]:
# @title What drives the selected model overall { display-mode: "form" }
shap_g, lime_g = gfi(champion_summary, "shap"), gfi(champion_summary, "lime")
if shap_g or lime_g:
    imp = pd.DataFrame({"SHAP": pd.Series(shap_g, dtype=float), "LIME": pd.Series(lime_g, dtype=float)}).fillna(0)
    imp = imp.div(imp.max().replace(0, 1))
    top = imp.max(axis=1).sort_values(ascending=False).head(12).index[::-1]
    fig = go.Figure([go.Bar(y=top, x=imp.loc[top, c], name=c, orientation="h", marker_color=col) for c, col in (("SHAP", BLUE), ("LIME", ORANGE)) if imp[c].any()])
    fig.update_layout(template="plotly_white", barmode="group", height=560, title=f"Global explanation from the platform · {CHAMPION_BUILD} (scaled to each method's top feature)")
    fig.show()
    drivers = imp.max(axis=1).sort_values(ascending=False).head(5).index.tolist()
else:
    print("No global SHAP or LIME stored for this build; using signal ranking for the drivers.")
    drivers = [c for c in MI.index if c in FEATURES][:5]
ss = support_rows(champion_summary)
if len(ss):
    display(ss.sort_values("influence", ascending=False).head(10).style.set_caption("Training rows that influenced predictions most (support-set attribution)"))
record("explainability", method="SHAP and LIME computed by the platform at training time; per-case SHAP/LIME through case_predict", top_drivers=", ".join(drivers))

# 5 · Model testing
The selected build scores the whole **Test** tag on platform compute. The scores come back as a table; the dashboards are drawn here.

In [ ]:
scored = project.model_inference(tag="Test", model_name=CHAMPION_BUILD, compute_type=COMPUTE)
scored[[UID, "Predicted_value_AutoML", "pred_proba_AutoML"]].head()

In [ ]:
# @title Evaluation dashboard on the held-out Test tag { display-mode: "form" }
PRED = "pred_proba_AutoML" if TASK == "classification" else "Predicted_value_AutoML"
truth = frames["Test"].set_index(frames["Test"][UID].astype(str))[TARGET_COL]
scored["_actual"] = scored[UID].astype(str).map(truth)
if TASK == "classification":
    p_te = scored["pred_proba_AutoML"].astype(float).to_numpy()
    y = (scored["_actual"].astype(str) == "1").astype(int).to_numpy()
    grid = np.linspace(0.05, 0.95, 91)
    THRESHOLD = float(grid[np.argmax([skm.f1_score(y, (p_te >= t).astype(int)) for t in grid])])
    perf = classification_dashboard(y, p_te, f"{CHAMPION_BUILD} · {len(y):,} held-out rows scored by the platform", threshold=THRESHOLD)
    perf = {k.lower().replace("@thr", "_at_threshold").replace("_%", "_pct"): float(v) for k, v in perf.items()}; perf["threshold"] = THRESHOLD
else:
    p_te = scored["Predicted_value_AutoML"].astype(float).to_numpy()
    y = scored["_actual"].astype(float).to_numpy()
    perf = regression_dashboard(y, p_te, f"{CHAMPION_BUILD} · {len(y):,} held-out rows scored by the platform")
    perf = {k.lower().replace("@thr", "_at_threshold").replace("_%", "_pct"): float(v) for k, v in perf.items()}
display(pd.Series(perf, name="held-out Test tag").to_frame().T.style.format(precision=3))
record("performance", evaluated_on="Test tag, scored by the platform with model_inference", test_rows=len(y), **perf)

# 6 · Model inference in batch mode
A new month arrives as a tag. One call scores it; the result is saved as a file for the downstream system.

In [ ]:
batch = project.model_inference(tag=MONTH_TAGS[0], model_name=CHAMPION_BUILD, compute_type=COMPUTE)      # a new month arrives → score the tag
batch.to_csv("batch_scored.csv", index=False)
batch[[UID, "Predicted_value_AutoML", "pred_proba_AutoML"]].head()

In [ ]:
# @title Batch output and the platform's inference history { display-mode: "form" }
col = "pred_proba_AutoML" if TASK == "classification" else "Predicted_value_AutoML"
fig = px.histogram(batch, x=col, nbins=40, color_discrete_sequence=[BLUE], title=f"Batch scoring on the platform · tag {MONTH_TAGS[0]} · {len(batch):,} rows → batch_scored.csv")
fig.update_layout(template="plotly_white", height=360); fig.show()
runs = attempt("inference history", lambda: project.model_inferences(), show=False)
display(as_frame(runs).head(5))

# 7 · Model risk

## 7.1 Uncertainty, and 7.2 fairness
Both are measured on the platform's own scores. **Conformal prediction** turns each score into a set (classification) or an interval (regression) with a coverage promise: half of the Test tag calibrates it, the other half checks it. **Fairness** compares groups on one attribute available in the data.

In [ ]:
# @title Uncertainty with measured coverage, and fairness by group { display-mode: "form" }
# Split conformal on the platform's own scores: half of the Test tag calibrates, the other half checks the promise
ALPHA = 0.10
rng_ = np.random.default_rng(0); idx = rng_.permutation(len(y)); cal, ev = idx[: len(y) // 2], idx[len(y) // 2:]
if TASK == "classification":
    score_true = np.where(y[cal] == 1, 1 - p_te[cal], p_te[cal])                 # how wrong the model was about the true class
    q = float(np.quantile(score_true, min(1.0, np.ceil((len(cal) + 1) * (1 - ALPHA)) / len(cal))))
    in0, in1 = (p_te[ev] <= q), ((1 - p_te[ev]) <= q)                            # is class 0 / class 1 inside the set?
    sizes_ = in0.astype(int) + in1.astype(int)
    covered = np.where(y[ev] == 1, in1, in0)
    unc = {"target_coverage": 1 - ALPHA, "measured_coverage": float(covered.mean()), "share_single_answer": float((sizes_ == 1).mean()),
           "share_needs_review": float((sizes_ != 1).mean())}
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Coverage: promised vs measured", "Prediction-set size"))
    vc = pd.Series(sizes_).value_counts().sort_index()
    fig.add_trace(go.Bar(x=[{0: "abstain", 1: "one answer", 2: "both possible"}[int(k)] for k in vc.index], y=vc.values, marker_color=BLUE, showlegend=False), 1, 2)
else:
    q = float(np.quantile(np.abs(y[cal] - p_te[cal]), min(1.0, np.ceil((len(cal) + 1) * (1 - ALPHA)) / len(cal))))
    covered = np.abs(y[ev] - p_te[ev]) <= q
    unc = {"target_coverage": 1 - ALPHA, "measured_coverage": float(covered.mean()), "interval_half_width": q,
           "interval_width_vs_median_target_pct": float(2 * q / np.median(y) * 100)}
    o = np.argsort(p_te[ev])[:: max(1, len(ev) // 120)]
    fig = make_subplots(rows=1, cols=2, column_widths=[0.3, 0.7], subplot_titles=("Coverage: promised vs measured", "90% intervals for a sample (sorted by prediction)"))
    xs = np.arange(len(o))
    fig.add_trace(go.Scatter(x=np.r_[xs, xs[::-1]], y=np.r_[p_te[ev][o] + q, (p_te[ev][o] - q)[::-1]], fill="toself", fillcolor="rgba(47,91,234,0.18)", line=dict(width=0), name="90% interval"), 1, 2)
    fig.add_trace(go.Scatter(x=xs, y=y[ev][o], mode="markers", marker=dict(color=ORANGE, size=5), name="actual"), 1, 2)
fig.add_trace(go.Bar(x=["promised", "measured"], y=[1 - ALPHA, unc["measured_coverage"]], marker_color=[GREY, TEAL], showlegend=False,
                     text=[f"{1 - ALPHA:.0%}", f"{unc['measured_coverage']:.1%}"], textposition="outside"), 1, 1)
fig.update_layout(template="plotly_white", height=400, title="Uncertainty: split conformal prediction on the platform's scores")
fig.show()
record("uncertainty", method="split conformal prediction on platform scores; half of the Test tag calibrates, half verifies", **unc)

# Fairness by group, from the platform's scores
gcols = [c for c in test.columns if c.startswith(GROUP_PREFIX)]
gser = (test[gcols].idxmax(axis=1).str.replace(GROUP_PREFIX, "") if len(gcols) > 1 else test[gcols[0]].astype(str)) if gcols else pd.Series("all", index=test.index)
group = scored[UID].astype(str).map(dict(zip(test[UID].astype(str), gser))).fillna("other").to_numpy()
if TASK == "classification":
    flag = (p_te >= THRESHOLD).astype(int)
    tab = pd.DataFrame({"g": group, "y": y, "flag": flag}).groupby("g").apply(lambda d: pd.Series({
        "rows": len(d), "base_rate": d.y.mean(), "selection_rate": d.flag.mean(), "true_positive_rate": d.flag[d.y == 1].mean(), "false_positive_rate": d.flag[d.y == 0].mean()}))
    fair = {"demographic_parity_difference": float(tab.selection_rate.max() - tab.selection_rate.min()),
            "disparate_impact_ratio": float(tab.selection_rate.min() / max(tab.selection_rate.max(), 1e-9)),
            "equal_opportunity_difference": float(tab.true_positive_rate.max() - tab.true_positive_rate.min())}
else:
    tab = pd.DataFrame({"g": group, "y": y, "p": p_te}).groupby("g").apply(lambda d: pd.Series({
        "rows": len(d), "mean_actual": d.y.mean(), "mean_prediction": d.p.mean(), "mae": (d.y - d.p).abs().mean()}))
    fair = {"mean_prediction_ratio": float(tab.mean_prediction.min() / tab.mean_prediction.max()), "mae_ratio": float(tab.mae.min() / tab.mae.max())}
tab = tab[tab.rows >= 30] if (tab.rows >= 30).sum() >= 2 else tab
display(tab.style.format(precision=3).set_caption(f"Fairness by {GROUP_LABEL}, computed from the platform's scores"))
record("fairness", attribute=GROUP_LABEL, groups=len(tab), **fair)
record("limitations", note_1="fairness is measured on one attribute available in the data; other protected attributes were not available",
       note_2="zero-shot foundation models depend on the uploaded context rows; refresh the Training tag when the population shifts",
       note_3="results come from a sample of the source data, not the full portfolio")

## 7.3 Observations and policies
Two platform guardrails sit next to the model and travel with every case:

- **Observations** are named facts about a case: an expression over its features plus a plain-English statement. They explain *why a case deserves attention* without changing the decision.
- **Policies** are rules that **override** the model's output with a decision when their expression is true (for example, always refer a case to manual review), in priority order, optionally for chosen models only.

Expressions use feature names, the operators `==`, `!==`, `>`, `<`, `in`, `notin`, `contains`, `notcontains`, the connectors `and` / `or`, and parentheses. Comparison values are whole words or whole numbers (the SDK's parser reads `\w+`, so no decimals or minus signs). Statements can quote a feature value with `{feature}`.

The cell below proposes observations and policies from the strongest features that survived data preparation; each is created through the SDK after a human gate and then appears on every case scored with `risk_policies=True` (section 8.2).

In [ ]:
# @title Propose observations and policies from the strongest features { display-mode: "form" }
_OPS = ("in", "notin", "contains", "notcontains")


def _threshold(feature, q):
    """A whole-number threshold the platform's expression parser accepts: no decimals, no minus sign, digits not inside the feature name."""
    v = int(np.floor(float(ctx[feature].quantile(q))))
    return v if v >= 0 and str(v) not in feature else None


cands = [c for c in MI.index if c in FEATURES and c in cont and not c.startswith("fe_")][:20]
OBS_SPECS = []
for f in cands:
    hi = _threshold(f, 0.90)
    share = float((ctx[f] > hi).mean()) if hi is not None else 0.0
    if hi is None or not 0.02 <= share <= 0.40:
        continue
    OBS_SPECS.append({"observation_name": f"High {f}"[:40], "expression": f"{f} > {hi}",
                      "statement": f"{f} is in the top {share:.0%} of the portfolio: {{{f}}}", "linked_features": [f]})
    if len(OBS_SPECS) == 3:
        break

POLICY_SPECS = []
pair = next(((a, b) for a in OBS_SPECS for b in OBS_SPECS if a is not b and not b["linked_features"][0].startswith(_OPS)), None)   # the parser reads a leading "in…" as an operator
if pair:
    a, b = pair; fa, fb = a["linked_features"][0], b["linked_features"][0]
    POLICY_SPECS.append({"policy_name": "Two risk flags together", "expression": f"{a['expression']} and {b['expression']}",
                         "statement": f"{fa} and {fb} are both unusually high: {{{fa}}}, {{{fb}}}", **POLICY_DECISION, "priority": 1})
if OBS_SPECS:
    f = OBS_SPECS[0]["linked_features"][0]; vh = _threshold(f, 0.98)
    if vh is not None and vh > int(OBS_SPECS[0]["expression"].split(">")[1]):
        POLICY_SPECS.append({"policy_name": f"Extreme {f}"[:40], "expression": f"{f} > {vh}", "statement": f"{f} is extreme: {{{f}}}",
                             **POLICY_DECISION, "priority": 2})
show = lambda specs, cols, cap: display(pd.DataFrame(specs)[cols].style.hide(axis="index").set_caption(cap)) if specs else print(f"{cap}: none proposed")
show(OBS_SPECS, ["observation_name", "expression", "statement"], "Observations proposed")
show([{**p, "decision": p.get("input") or p["decision"]} for p in POLICY_SPECS], ["policy_name", "expression", "decision", "priority"], "Policies proposed")

In [ ]:
params = attempt("expression parameters (features, operators, decisions)", lambda: project.get_observation_params(), default={})
have = set(as_frame(attempt("observations", lambda: project.observations(), show=False)).get("observation_name", []))
for spec in OBS_SPECS:
    if spec["observation_name"] not in have and human_gate(f"Create observation '{spec['observation_name']}' ({spec['expression']})?", True):
        attempt(f"create_observation · {spec['observation_name']}", lambda s=spec: project.create_observation(**s))
as_frame(attempt("observations", lambda: project.observations(), show=False))

In [ ]:
have = set(as_frame(attempt("policies", lambda: project.policies(), show=False)).get("policy_name", []))
for spec in POLICY_SPECS:
    if spec["policy_name"] not in have and human_gate(f"Create policy '{spec['policy_name']}' ({spec['expression']})?", True):
        attempt(f"create_policy · {spec['policy_name']}", lambda s=spec: project.create_policy(models=[CHAMPION_BUILD], **s))
as_frame(attempt("policies", lambda: project.policies(), show=False))

## 7.4 Regulatory evidence map

In [ ]:
# @title Which evidence supports which expectation { display-mode: "form" }
REG_MAP = pd.DataFrame([
    ("Conceptual soundness", "Problem statement, data preparation log, feature rationale, model selection rule", "SR 11-7 · SAMA model risk expectations"),
    ("Data quality and lineage", "Data profile, agent action log with human approvals, leakage checks", "SAMA Data Management · NDMO standards · PDPL"),
    ("Outcomes analysis", "Hold-out metrics, evaluation dashboards, segment performance", "SR 11-7 · Basel sound-practice principles"),
    ("Uncertainty", "Conformal prediction with measured coverage, calibration", "SR 11-7 (model limitations) · CBB AI policy"),
    ("Fairness", "Group metrics on protected or proxy attributes", "SDAIA AI Ethics Principles · CBB AI policy · EU AI Act"),
    ("Explainability", "Global drivers and per-decision explanations", "SDAIA AI Ethics Principles · EU AI Act"),
    ("Ongoing monitoring", "Drift monitoring, performance tracking, retraining with approval gate", "SR 11-7 · SAMA model risk expectations"),
    ("Documentation", "Generated model card with grounding check, audit trail", "SR 11-7 · ISO/IEC 42001"),
], columns=["Expectation", "Evidence produced in this notebook", "Frameworks it speaks to"])
display(REG_MAP.style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "pre-wrap"})
        .set_caption("Frameworks are named for orientation only. The bank's Compliance and Risk functions own the control mapping; "
                     "no clause-level compliance is claimed here."))
n_obs = len(as_frame(attempt("observations", lambda: project.observations(), show=False)))
n_pol = len(as_frame(attempt("policies", lambda: project.policies(), show=False)))
record("governance", frameworks_referenced="SR 11-7; SAMA model risk expectations; SAMA and NDMO data standards; PDPL; CBB AI policy; SDAIA AI Ethics Principles; EU AI Act; ISO/IEC 42001",
       platform_observations=n_obs, platform_policies=n_pol, human_approval_gates=sum(1 for a in AUDIT if a["actor"] == "human"),
       llm_reviewed_agent_steps=sum(1 for t in AGENT_TRACE if t["decision"] in ("approve", "reject")))

# 8 · Artefacts before production

## 8.1 Model card through the gateway
The card is written by the `card_writer` model **only from the facts this notebook recorded**, against a documentation standard written for a model-risk committee: a system prompt (role, citation and number rules, what each of the 14 sections must contain, a self-check), the card template, the fact sheet with a citation key per fact, and metric notes. Automatic checks then verify every number against its fact, and the issue list goes back to the writer for one revision. Only the aggregate fact sheet is sent, never customer rows.

In [ ]:
# @title The documentation standard: system prompt, 14-section template, metric notes { display-mode: "form" }
import html as _html, textwrap
REVISION_PASS = True            # if the automatic checks find problems, send them back to the writer once

CARD_SECTIONS = [
    "0. Document control", "1. Executive summary", "2. Intended use and scope", "3. Data", "4. Methodology and model selection",
    "5. Performance on held-out data", "6. Explainability", "7. Uncertainty and calibration", "8. Fairness",
    "9. Limitations and known risks", "10. Monitoring, alerting and retraining", "11. Regulatory alignment",
    "12. Approvals and sign-off", "13. Change log",
]

SYSTEM_PROMPT = textwrap.dedent("""
# ROLE
You are a senior model-risk documentation specialist at a Gulf bank supervised by SAMA (Saudi Central Bank) and the
Central Bank of Bahrain. You write the model card that goes to three readers:
- the model-risk committee, who decide whether the model may be used and read sections 0, 1, 9 and 12 first;
- the independent validator, who checks every number against the evidence;
- the supervisor's examiner, who looks for honest limitations and working controls.
A card that overstates, invents or cannot be traced is worse than no card. Precision and traceability come before style.

# SOURCE OF TRUTH
The user message contains a FACT SHEET. It is the only source you may use. Each fact has a key, for example
`performance.roc_auc = 0.7277`. The part before the dot is the fact group (data, preparation, selection, performance ...).

# CITATION RULES
1. Every number you write is followed immediately by the key it came from, in square brackets.
   Right:  The model reaches a ROC AUC of 0.728 [performance.roc_auc] on 3,000 unseen loans [performance.test_rows].
   Wrong:  The model reaches a ROC AUC of about 0.73 on the test set.            (no key; "about")
   Wrong:  ROC AUC improved by 0.015 over the baseline.                           (a number you computed)
2. Inside a table, put the key in the "Source" column of the same row instead of in brackets.
3. Use a key only for the fact it names. Never attach a key to a different number.
4. Text facts (model names, feature names, methods, rules) are quoted as written and also carry their key.

# NUMBER RULES
5. Round, never change: scores to 3 decimals, counts to whole numbers with thousands separators, percentages to 1 decimal.
6. A fact stored as a share (0.9017) may be written as a percentage (90.2%) with the same key. Say which it is.
7. Do NO arithmetic. Do not write differences, ratios, sums, averages or percentage changes that are not themselves facts.
   To compare two facts, state both with their keys and say which is higher, lower or about equal.
8. If something the template asks for is not in the fact sheet, write exactly "Not recorded". Never estimate or infer it.

# NEVER WRITE
9. Dates, people's names, team names, committee names, ticket numbers or version numbers that are not in the fact sheet.
10. Approvals, validation outcomes or sign-offs. Every approval is "Pending". The risk tier is "To be assigned by Model Risk Management".
11. Regulation article, clause or circular numbers, and any statement that the model "complies", "is compliant" or "meets"
    a regulation. Name a framework only if it appears in the fact sheet, and describe the evidence, not compliance.
12. Marketing language: "state-of-the-art", "robust", "best-in-class", "ensures", "guarantees", "cutting-edge".
13. Advice the facts do not support (for example recommending a threshold or a new feature).

# WHAT EACH SECTION MUST CONTAIN
0 Document control: the table, filled from use_case.* and the values marked "given".
1 Executive summary: exactly five bullets, in this order: what the model decides and for whom; which model was selected and
  how (selection.*); the headline held-out result with its metric explained in a few words (performance.*); how uncertainty
  is handled (uncertainty.*); the single most important risk to watch (limitations.* or monitoring.*). No bullet over 35 words.
2 Intended use and scope: the decision supported, intended users (only if recorded), the population the data covers, and an
  "Out of scope" list of at least three uses the facts do not support (other products, other populations, automated decisions
  without the recorded human review, and so on, phrased as limits of the evidence).
3 Data: source and size; a table of the data sets with rows and purpose; candidate features and what remained after
  preparation; what the data-preparation agent removed and why, including leakage controls (preparation.*); data-quality
  findings (missing values, skew); engineered features (features.*).
4 Methodology and model selection: how many candidates were raced and how many succeeded; the selection rule in plain words;
  the selected model and whether it is zero-shot, fine-tuned or trained; the hyperparameter search; why it won
  (selection.selection_reason if recorded); the hardware (training.*).
5 Performance on held-out data: the table with one row per performance.* fact and a plain-English "What it means" taken from
  the METRIC NOTES; then two or three sentences comparing the selected model with the best classic baseline (state both
  numbers, no arithmetic) and saying what the result means for the decision.
6 Explainability: the global method and the top drivers named in plain words; how single decisions are explained; a caution
  that importance shows association with the prediction, not cause.
7 Uncertainty and calibration: the method; promised versus measured coverage; what happens to uncertain cases; calibration
  figures if recorded.
8 Fairness: the table with one row per fairness.* fact; the attribute used and the number of groups; a neutral reading of the
  largest gap; a statement that other protected attributes were not assessed if the facts say so. No legal conclusions.
  For forecasting models write that fairness metrics do not apply and report consistency across segments instead.
9 Limitations and known risks: a numbered list of at least four items. Each item has three parts: the limitation, why it
  matters for the decision, and the control or monitor that addresses it (from monitoring.*, uncertainty.* or governance.*),
  or "No control recorded".
10 Monitoring, alerting and retraining: the monitors; the alert thresholds if recorded; how alerts reach the retraining
  agent; the champion-challenger rule and promotion margin; every human approval gate; then what monitoring has observed so
  far and every retraining event (retraining.*). If there are no observations yet, say "No production observations yet".
11 Regulatory alignment: the table. For each expectation list the card sections that hold the evidence, write the status as
  "Evidence provided" or "Not recorded", and cite one fact key. Close with one sentence naming the frameworks in
  governance.frameworks_referenced and stating that control mapping is owned by the bank's Compliance function.
12 Approvals and sign-off: the table with every cell "Pending", followed by "Open items for the validator": a bullet for
  every "Not recorded" you wrote in the card, or "None" if there were none.
13 Change log: every change supplied by the user, as bullets in the order given, or "Initial version".

# STYLE
- Plain English, short sentences, active voice. Explain each metric in a few words the first time it appears.
- Prefer tables for numbers and bullets for lists. No paragraph longer than four sentences.
- Feature names are written as recorded, in backticks.
- Target length: 900 to 1,500 words excluding tables.

# BEFORE YOU ANSWER, CHECK
- Every number has a key next to it or in its row, and the key exists in the fact sheet.
- No number was computed by you. No date, name, approval or clause number was invented.
- All 14 headings are present, numbered 0 to 13, in order, exactly as in the template.
- Every "Not recorded" in the card is listed under section 12.

# EXTRA RULES
- Square brackets contain ONLY a fact key, nothing else. Never write notes, reasoning or "not recorded" inside brackets.
  If the exact fact is missing, use the closest fact that is recorded with its own key, or write "Not recorded" in plain text.
- In section 11 the status may be "Evidence provided", "Partly provided: <which part>", "Not applicable: <why>" or "Not recorded".
  Use "Not recorded" only when no section of the card holds any evidence for that expectation.

# OUTPUT
Markdown only. Start with the line "# Model card · <use-case name>" and end with the last line of section 13.
No preamble, no closing remarks, no code fence around the card.
""").strip()

CARD_TEMPLATE = textwrap.dedent("""
# Model card · <use-case name>

## 0. Document control
| Field | Value | Source |
|---|---|---|
| Model name | <use-case name> | use_case.name |
| Model version | <version> | use_case.version |
| Card version | <card version given by the user> | given |
| Status | Draft, pending independent validation | given |
| Model owner | <owner> | use_case.owner |
| Risk tier | To be assigned by Model Risk Management | given |
| Written by | <writer given by the user> | given |

## 1. Executive summary
<five bullets as specified>

## 2. Intended use and scope
<decision supported, intended users, population in scope>

**Out of scope**
<at least three bullets>

## 3. Data
<source and size in one or two sentences>

| Data set | Rows | Used for | Source |
|---|---|---|---|
<one row per recorded data set>

<features before and after preparation; what the agent removed and why; leakage controls; data-quality findings; engineered features>

## 4. Methodology and model selection
<candidates, selection rule, selected model, adaptation, hyperparameter search, why it won, hardware>

## 5. Performance on held-out data
| Metric | Value | What it means | Source |
|---|---|---|---|
<one row per recorded performance fact>

<comparison with the best classic baseline, and what the result means for the decision>

## 6. Explainability
<global method and top drivers; single-decision explanations; the association-not-cause caution>

## 7. Uncertainty and calibration
<method, promised vs measured coverage, treatment of uncertain cases, calibration>

## 8. Fairness
| Metric | Value | Source |
|---|---|---|
<one row per recorded fairness fact, or a single row "Not recorded">

<neutral interpretation and limits>

## 9. Limitations and known risks
<numbered list; each item: limitation · why it matters · control or monitor>

## 10. Monitoring, alerting and retraining
<monitors, thresholds, alert routing, champion-challenger rule, approval gates>

**Observed so far**
<observations and retraining events, or "No production observations yet">

## 11. Regulatory alignment
| Expectation | Evidence in this card | Status | Source |
|---|---|---|---|
| Conceptual soundness | <sections> | <status> | <key> |
| Data quality and lineage | <sections> | <status> | <key> |
| Outcomes analysis | <sections> | <status> | <key> |
| Ongoing monitoring | <sections> | <status> | <key> |
| Explainability and fairness | <sections> | <status> | <key> |
| Documentation and change control | <sections> | <status> | <key> |

<one sentence naming the frameworks in the fact sheet; control mapping is owned by the bank's Compliance function>

## 12. Approvals and sign-off
| Role | Name | Decision | Date |
|---|---|---|---|
| Model owner | Pending | Pending | Pending |
| Independent model validation | Pending | Pending | Pending |
| Model risk committee | Pending | Pending | Pending |

**Open items for the validator**
<one bullet per "Not recorded" in this card, or "None">

## 13. Change log
<bullets>
""").strip()

# Plain-English meanings the writer may use in the "What it means" column (deliberately free of numbers)
METRIC_NOTES = {
    "roc_auc": "how well the model ranks risky cases above safe ones; higher is better",
    "average_precision": "how concentrated the true positives are among the highest scores; higher is better",
    "f1": "balance of precision and recall at the chosen threshold; higher is better",
    "precision": "of the cases the model flags, the share that are truly positive",
    "recall": "of the truly positive cases, the share the model flags",
    "accuracy": "share of all cases classified correctly at the chosen threshold",
    "balanced_accuracy": "accuracy averaged over the two classes, so the rarer class counts equally",
    "mcc": "single summary of the confusion matrix that stays fair when classes are unbalanced; higher is better",
    "brier": "average squared gap between predicted probability and outcome; lower is better",
    "log_loss": "penalty for confident wrong probabilities; lower is better",
    "ks": "largest separation between the score distributions of the two classes; higher is better",
    "gini": "a rescaling of ROC AUC used in credit scoring; higher is better",
    "threshold": "the score above which a case is flagged; chosen on validation data",
    "r2": "share of the variation in the target that the model explains; higher is better",
    "rmse": "typical size of the error, with large errors weighted more; in the target's units; lower is better",
    "mae": "average size of the error in the target's units; lower is better",
    "median_abs_error": "the error size that half of the predictions stay within; lower is better",
    "mape_pct": "average error as a percentage of the actual value; lower is better",
    "explained_variance": "share of the target's variation captured by the predictions; higher is better",
    "max_error": "the single largest error on the test set",
    "within_20pct_share": "percentage of predictions that land within one fifth of the actual value",
    "mase": "forecast error relative to a seasonal-naive forecast; below one beats it; lower is better",
    "wql": "error of the whole predicted range, not just the middle; lower is better",
    "smape": "symmetric percentage error of the median forecast; lower is better",
    "wape_pct": "total absolute error as a percentage of total actual flow; lower is better",
    "inside_90_band_pct": "percentage of actual outcomes that fell inside the ninety-percent band",
    "inside_80_band_pct": "percentage of actual outcomes that fell inside the eighty-percent band",
    "batch_rows_per_second": "scoring throughput in batch mode",
    "measured_coverage": "share of test cases whose true outcome fell inside the prediction set or interval",
    "disparate_impact_ratio": "lowest group selection rate divided by the highest; closer to one means more equal",
    "demographic_parity_difference": "gap between the highest and lowest group selection rates; closer to zero is more equal",
    "equal_opportunity_difference": "gap between groups in the share of true positives that are flagged",
    "equalized_odds_difference": "the larger of the gaps in true-positive and false-positive rates between groups",
    "predictive_parity_difference": "gap between groups in the precision of the flag",
    "max_feature_psi": "largest population stability index of any monitored feature; higher means more drift",
}

SECTION_GUIDE = {"use_case": "0, 1, 2", "data": "3", "preparation": "3", "features": "3", "selection": "4", "training": "4",
                 "performance": "1, 5", "explainability": "6", "uncertainty": "7", "fairness": "8", "limitations": "9",
                 "monitoring": "10", "retraining": "10, 13", "governance": "11"}


def flatten_facts(facts):
    flat = {}
    for sec, vals in facts.items():
        for k, v in vals.items():
            if isinstance(v, (np.floating, np.integer)):
                v = float(v)
            if isinstance(v, float) and not np.isfinite(v):
                continue
            flat[f"{sec}.{k}"] = v
    return flat


def build_messages(facts, changes, version_label, writer_label):
    flat = flatten_facts(facts)
    groups = {}
    for k, v in flat.items():
        groups.setdefault(k.split(".", 1)[0], []).append(f"- {k} = {v}")
    fact_block = "\n\n".join(f"[{g}]  (mainly for section {SECTION_GUIDE.get(g, 'as relevant')})\n" + "\n".join(lines) for g, lines in groups.items())
    notes = "\n".join(f"- {k}: {METRIC_NOTES[k.split('.', 1)[1]]}" for k in flat if k.split(".", 1)[1] in METRIC_NOTES)
    change_lines = "\n".join(f"- {c}" for c in changes) if changes else "- Initial version"
    user = (f"Write the model card, card version {version_label}. It is being written by: {writer_label}.\n\n"
            f"=== TEMPLATE (reproduce the headings and tables exactly; replace each <instruction>) ===\n\n{CARD_TEMPLATE}\n\n"
            f"=== FACT SHEET ({len(flat)} facts; the only source you may use; cite the key after every number) ===\n\n{fact_block}\n\n"
            f"=== METRIC NOTES (plain-English meanings for the 'What it means' column) ===\n{notes or '- none'}\n\n"
            f"=== CHANGES SINCE THE PREVIOUS VERSION (for section 13) ===\n{change_lines}\n")
    return SYSTEM_PROMPT, user


def writer_label():
    return f"{LEXSI_PROVIDER} · {LEXSI_MODEL} through the Lexsi LLM gateway" if WRITER == "lexsi" else (
        "open-weights model on the notebook GPU" if WRITER not in ("template",) else "template writer (no LLM)")


def show_prompt(version_label="v1", changes=None):
    system, user = build_messages(FACTS, changes, version_label, writer_label())
    display(Markdown(f"**Prompt sent to the writer:** system prompt {len(system):,} characters · user message {len(user):,} characters "
                     f"(about {(len(system) + len(user)) // 4:,} tokens) · {len(flatten_facts(FACTS))} facts · no customer-level data."))
    for title, body in (("System prompt: role, citation rules, section requirements, self-check", system),
                        ("User message: template, fact sheet, metric notes, changes", user)):
        display(HTML(f"<details><summary><b>{title}</b> · click to expand</summary>"
                     f"<pre style='white-space:pre-wrap;font-size:12px'>{_html.escape(body)}</pre></details>"))

In [ ]:
# @title Writer through the gateway, template fallback and the automatic checks { display-mode: "form" }
# The model card is written through the same gateway, by the model assigned to the "card_writer" role
LEXSI_PROVIDER, LEXSI_MODEL = GATEWAY_MODELS["card_writer"]
LEXSI_PROJECT = LLM_PROJECT_ID
WRITER = "lexsi" if gateway_ready() else "template"


def write_lexsi(system, messages):
    max_tokens = CARD_MAX_TOKENS
    for attempt_ in range(2):
        res = llm_call("card_writer", [{"role": "system", "content": system}, *messages], max_tokens=max_tokens, agent="model-card writer",
                       step="revision" if len(messages) > 1 else "draft")
        if res["finish_reason"] in ("length", "max_tokens") and attempt_ == 0:
            print(f"answer cut off at {max_tokens:,} tokens → asking again with {max_tokens * 2:,}")
            max_tokens *= 2
            continue
        break
    text = res["text"]
    if "# Model card" in text:
        text = text[text.index("# Model card"):]
    text = re.sub(r"^```(?:markdown)?\s*|\s*```$", "", text.strip())
    AGENT_TRACE[-1].update(decision="card written", rationale=f"{len(text):,} characters")
    return text, res["reasoning"], {"writer": f"{res['provider']} · {res['model']} (Lexsi gateway, via {res['via']})", "max_tokens_used": max_tokens,
                                    "input_tokens": res["input_tokens"], "output_tokens": res["output_tokens"], "finish_reason": res["finish_reason"],
                                    "lexsi_project": LLM_PROJECT_ID, "gateway_seconds": res["seconds"]}

print("model-card writer:", f"{LEXSI_PROVIDER} · {LEXSI_MODEL} through the Lexsi gateway" if WRITER == "lexsi" else "template (no gateway configured)")
write_local = None


def write_template(facts, changes, version_label):
    """Deterministic writer: fills the same template from the facts, with a citation after every number."""
    flat = flatten_facts(facts)
    def cite(key, label=None):
        v = flat.get(key)
        if v is None:
            return "Not recorded"
        if isinstance(v, float):
            v = f"{v:,.4g}" if abs(v) < 1000 else f"{v:,.0f}"
        return f"{v} [{key}]"
    rows = lambda sec: [(k.split(".", 1)[1], k) for k in flat if k.startswith(sec + ".")]
    table = lambda sec: "\n".join(f"| {name.replace('_', ' ')} | {cite(k)} | {k} |" for name, k in rows(sec)) or "| Not recorded | Not recorded | — |"
    uc = lambda k: flat.get(f"use_case.{k}", "Not recorded")
    L = [f"# Model card · {uc('name')}", "", "## 0. Document control", "| Field | Value | Source |", "|---|---|---|",
         f"| Model name | {uc('name')} | use_case.name |", f"| Model version | {uc('version')} | use_case.version |",
         f"| Card version | {version_label} | given |", "| Status | Draft, pending independent validation | given |",
         f"| Model owner | {uc('owner')} | use_case.owner |", "| Risk tier | To be assigned by Model Risk Management | given |",
         "| Written by | template writer (no LLM) | given |", "",
         "## 1. Executive summary", f"- Decision supported: {uc('decision_supported')}.",
         f"- Selected model: {cite('selection.selected_model')}.", f"- Selection metric: {cite('selection.selection_metric')}.",
         "- Held-out performance: see section 5.", "- Main risk to watch: drift in the population; see sections 9 and 10.", "",
         "## 2. Intended use and scope", f"- Task: {uc('task')}.", f"- Decision supported: {uc('decision_supported')}.",
         "- Out of scope: any use not described in the facts.", "",
         "## 3. Data", "| Item | Value | Source |", "|---|---|---|", table("data"), table("preparation"), table("features"), "",
         "## 4. Methodology and model selection", "| Item | Value | Source |", "|---|---|---|", table("selection"), table("training"), "",
         "## 5. Performance on held-out data", "| Metric | Value | What it means | Source |", "|---|---|---|---|",
         "\n".join(f"| {n.replace('_', ' ')} | {cite(k)} | — | {k} |" for n, k in rows("performance")) or "| Not recorded | — | — | — |", "",
         "## 6. Explainability", "| Item | Value | Source |", "|---|---|---|", table("explainability"), "",
         "## 7. Uncertainty and calibration", "| Item | Value | Source |", "|---|---|---|", table("uncertainty"), "",
         "## 8. Fairness", "| Metric | Value | Source |", "|---|---|---|", table("fairness"), "",
         "## 9. Limitations and known risks", *[f"{i}. {flat[k]}" for i, k in enumerate([k for k in flat if k.startswith("limitations.")], 1)], "",
         "## 10. Monitoring, alerting and retraining", "| Item | Value | Source |", "|---|---|---|", table("monitoring"), table("retraining"), "",
         "## 11. Regulatory alignment", "| Expectation | Evidence in this card | Status | Source |", "|---|---|---|---|",
         "| Conceptual soundness | Sections 3–4 | Evidence provided | selection.selection_rule |",
         "| Data quality and lineage | Section 3 | Evidence provided | preparation.features_removed |",
         "| Outcomes analysis | Section 5 | Evidence provided | performance.evaluated_on |",
         "| Ongoing monitoring | Section 10 | Evidence provided | monitoring.plan |",
         "| Explainability and fairness | Sections 6, 8 | Evidence provided | explainability.method |",
         "| Documentation and change control | Sections 12–13 | Evidence provided | governance.human_approval_gates |", "",
         f"Frameworks referenced: {flat.get('governance.frameworks_referenced', 'Not recorded')}. Control mapping is owned by the bank's Compliance function.", "",
         "## 12. Approvals and sign-off", "| Role | Name | Decision | Date |", "|---|---|---|---|",
         "| Model owner | Pending | Pending | Pending |", "| Independent model validation | Pending | Pending | Pending |",
         "| Model risk committee | Pending | Pending | Pending |", "",
         "## 13. Change log", *([f"- {c}" for c in changes] if changes else ["- Initial version"])]
    return "\n".join(L), "", {"writer": "template writer (no LLM)"}

FORBIDDEN = [(r"\b(article|clause|circular)\s+(no\.?\s*)?\d", "cites a regulation article or clause number"),
             (r"\b(approved|validated|signed off)\s+(by|on)\b", "claims an approval or validation"),
             (r"\b(fully|is)\s+compliant\b|\bcomplies with\b", "claims compliance"),
             (r"\b(19|20)\d{2}-\d{2}-\d{2}\b", "contains a date that is not in the facts")]


def _matches(value, written):
    """Does the number as written match the fact value (allowing rounding and share↔percent)?"""
    try:
        v = float(value)
    except (TypeError, ValueError):
        return False
    d = len(written.split(".")[1]) if "." in written else 0
    n = float(written)
    tol = 0.5 * 10 ** (-d) + 1e-9
    return abs(v - n) <= tol or abs(v * 100 - n) <= tol or (abs(v) >= 1000 and abs(v - n) <= max(tol, abs(v) * 0.0005))


def check_card(card, facts, changes=None):
    """Automatic hallucination and completeness checks on a generated card."""
    flat = flatten_facts(facts)
    card = re.sub(r"\b[Ss]ections?\s+\d+(?:\s*[–-]\s*\d+)?(?:\s*(?:,|and)\s*\d+)*", "Section", card)   # section references are not facts
    text = card.replace(",", "")
    issues = []
    # 1. every cited key exists, and the number next to it matches the fact
    cites = list(re.finditer(r"\[([a-z_]+\.[a-z0-9_]+)\]", card))
    bad_keys = sorted({m.group(1) for m in cites if m.group(1) not in flat})
    mism = []
    for m in cites:
        k = m.group(1)
        if k not in flat:
            continue
        before = card[max(0, m.start() - 40):m.start()].replace(",", "")
        nums = re.findall(r"-?\d+(?:\.\d+)?", before)
        if nums and isinstance(flat[k], (int, float)) and not isinstance(flat[k], bool) and not _matches(flat[k], nums[-1]):
            mism.append(f"{nums[-1]} [{k}] (fact = {flat[k]})")
    # table rows: value cell vs source cell
    for row in re.findall(r"^\|(.+)\|\s*$", card, flags=re.M):
        cells = [c.strip() for c in row.split("|")]
        if len(cells) >= 3 and cells[-1] in flat and isinstance(flat[cells[-1]], (int, float)) and not isinstance(flat[cells[-1]], bool):
            nums = re.findall(r"-?\d+(?:\.\d+)?", " ".join(cells[1:-1]).replace(",", ""))
            if nums and not any(_matches(flat[cells[-1]], n) for n in nums):
                mism.append(f"table row '{cells[0]}': {nums[0]} vs {cells[-1]} = {flat[cells[-1]]}")
    # 2. numbers that are not grounded in any fact (prose only; ignore headings, list numbering and small integers)
    known = [v for v in flat.values() if isinstance(v, (int, float)) and not isinstance(v, bool)]
    known += [float(n) for v in flat.values() if isinstance(v, str) for n in re.findall(r"\d+(?:\.\d+)?", v.replace(",", ""))]
    known += [float(n) for k in flat for n in re.findall(r"\d+", k)]          # numbers that are part of a fact's name, e.g. the 80% band
    body = re.sub(r"^#+ .*$", "", text, flags=re.M)
    body = re.sub(r"^\s*\d+\.\s", "", body, flags=re.M)
    nums = [n for n in re.findall(r"(?<![\w.\[])-?\d+(?:\.\d+)?(?![\w\]])", body) if not (n.isdigit() and int(n) <= 13)]
    change_nums = set(re.findall(r"\d+(?:\.\d+)?", " ".join(changes or []).replace(",", "")))
    ungrounded = sorted({n for n in nums if n not in change_nums and not any(_matches(v, n) for v in known)})
    # 3. structure and forbidden content
    missing_sections = [s for s in CARD_SECTIONS if not re.search(r"^##\s*" + re.escape(s), card, flags=re.M)]
    fact_text = " ".join(str(v) for v in flat.values()).lower()
    forbidden = [why for pat, why in FORBIDDEN
                 if any(m.group(0).lower() not in fact_text for m in re.finditer(pat, card, flags=re.I))]     # allowed if it is a recorded fact
    notes = [m.group(0) for m in re.finditer(r"\[([^\]\n]{1,300})\]", card)
             if not re.fullmatch(r"[a-z_]+\.[a-z0-9_]+", m.group(1)) and "." in m.group(1) and "_" in m.group(1)]
    if notes: issues.append(f"brackets must contain only a fact key; rewrite these as plain text with a proper key: {'; '.join(n[:100] for n in notes[:5])}")
    if bad_keys: issues.append(f"cites keys that do not exist: {', '.join(bad_keys[:8])}")
    if mism: issues.append(f"numbers that do not match their cited fact: {'; '.join(mism[:8])}")
    if ungrounded: issues.append(f"numbers not found in the facts: {', '.join(ungrounded[:12])}")
    if missing_sections: issues.append(f"missing sections: {', '.join(missing_sections)}")
    issues += [f"forbidden content: {f}" for f in forbidden]
    report = {"citations": len(cites), "invalid_citation_keys": len(bad_keys), "malformed_citations": len(notes), "number_mismatches": len(mism),
              "ungrounded_numbers": len(ungrounded), "sections_present": f"{len(CARD_SECTIONS) - len(missing_sections)}/{len(CARD_SECTIONS)}",
              "forbidden_content": len(forbidden)}
    checked = max(len(nums) + len(cites), 1)
    report["grounded_pct"] = round(100 * (1 - (len(ungrounded) + len(mism)) / checked), 1)
    report["passed"] = not issues
    return report, issues


def generate_model_card(facts, changes=None, version_label="v1"):
    """Draft → automatic checks → one revision with the issue list (if needed) → final checks."""
    facts = json.loads(json.dumps(facts, default=float))
    trace, rounds = {"card_version": version_label}, []
    system, user = build_messages(facts, changes, version_label, writer_label())
    messages = [{"role": "user", "content": user}]
    write = write_lexsi if WRITER == "lexsi" else write_local
    t0 = time.time()
    try:
        if WRITER == "template":
            card, reasoning, meta = write_template(facts, changes, version_label)
        else:
            card, reasoning, meta = write(system, messages)
    except Exception as exc:
        print(f"⚠️ writer failed: {str(exc)[:500]}\n   → using the template writer for this card")
        card, reasoning, meta = write_template(facts, changes, version_label)
    report, issues = check_card(card, facts, changes)
    rounds.append({"round": "draft", **report})
    if issues and REVISION_PASS and not meta["writer"].startswith("template"):
        revise = ("An automatic checker compared your draft with the fact sheet and found these problems:\n- " + "\n- ".join(issues) +
                  "\n\nRewrite the COMPLETE card. Fix every problem listed: replace any number that is not in the fact sheet with the correct "
                  "fact and its key, or with 'Not recorded'; remove anything you computed yourself; restore any missing heading exactly as in "
                  "the template. Keep everything that was already correct. Same template, same rules, Markdown only.")
        messages += [{"role": "assistant", "content": card}, {"role": "user", "content": revise}]
        try:
            card2, reasoning2, meta2 = write(system, messages)
            report2, issues2 = check_card(card2, facts, changes)
            rounds.append({"round": "revision", **report2})
            if len(issues2) <= len(issues):
                card, issues, report = card2, issues2, report2
                reasoning = (reasoning + "\n\n--- revision ---\n" + reasoning2).strip()
                meta["output_tokens"] = meta.get("output_tokens", 0) + meta2.get("output_tokens", 0)
        except Exception as exc:
            print("revision pass failed:", str(exc)[:160])
    trace.update(meta, seconds=round(time.time() - t0, 1), rounds=len(rounds), prompt_characters=len(system) + len(user))
    trace["_reasoning"], trace["_rounds"], trace["_issues"] = reasoning, rounds, issues
    return card, trace, report


def show_card(card, trace, report):
    display(pd.DataFrame([{k: v for k, v in trace.items() if not k.startswith("_")}]).T.rename(columns={0: "trace"}))
    display(pd.DataFrame(trace["_rounds"]).set_index("round").T.rename_axis("automatic checks"))
    fig = go.Figure(go.Indicator(mode="gauge+number", value=report["grounded_pct"], number={"suffix": "%"},
                                 title={"text": "Numbers in the card that trace back to a recorded fact"},
                                 gauge={"axis": {"range": [0, 100]}, "bar": {"color": TEAL if report["passed"] else ORANGE}}))
    fig.update_layout(height=250, margin=dict(t=60, b=10)); fig.show()
    if trace["_issues"]:
        print("remaining issues for a human reviewer:\n- " + "\n- ".join(trace["_issues"]))
    if trace["_reasoning"]:
        display(HTML(f"<details><summary><b>Writer's reasoning trace</b> ({len(trace['_reasoning']):,} characters) · click to expand</summary>"
                     f"<pre style='white-space:pre-wrap;font-size:12px'>{_html.escape(trace['_reasoning'][:12000])}</pre></details>"))
    display(Markdown(card))

In [ ]:
record("use_case", name="Home Credit · Loan Amount Recommendation", task=TASK, decision_supported="recommended credit amount with a range", owner="[model owner]", version="v1")
show_prompt("v1")               # exactly what the writer receives

In [ ]:
CARD_V1, TRACE_V1, CHECKS_V1 = generate_model_card(FACTS, version_label="v1")

In [ ]:
show_card(CARD_V1, TRACE_V1, CHECKS_V1)
open("model_card_v1.md", "w").write(CARD_V1);

## 8.2 One case, explained, with observations and policies
`case_predict` scores a single case and returns its explanation. With `risk_policies=True` the platform also evaluates the observations and policies from section 7.3 on that case. This is the call a production system makes for each decision.

In [ ]:
CASE_UID = str(scored.sort_values(PRED, ascending=False)[UID].iloc[0])                    # the highest-scored case in the Test tag
case_xai = [m for m in (xai_for(CHAMPION, TASK) if CHAMPION in CATALOGUE.index else ["lime"]) if m != "support_set"]
case = attempt("case_predict with explanations, observations and policies", lambda: project.case_predict(
    unique_identifier=CASE_UID, tag="Test", model_name=CHAMPION_BUILD, xai=case_xai, risk_policies=True))
case.inference_output() if case is not None else None

In [ ]:
# @title Why this prediction, which observations fired, which policies applied { display-mode: "form" }
if case is not None:
    sh, li = case_attributions(case, "shap"), case_attributions(case, "lime")
    if sh or li:
        signed_bar(sh or li, f"Why this prediction? Case {CASE_UID} · {'SHAP' if sh else 'LIME'} from the platform").show()
    for title, fn in (("Observations that fired on this case", lambda: case.xai_observations()), ("Policies evaluated on this case", lambda: case.xai_policies())):
        df_ = as_frame(attempt(title, fn, show=False))
        display(df_.head(10).style.set_caption(title) if len(df_) else Markdown(f"*{title}: none returned.*"))

# 9 · Model in production

## 9.1 Monitoring on the platform
The production months were uploaded as tags (`ProdM1`, `ProdM2`, …). The platform compares any tag with the Training tag.

| What | SDK call | Result |
|---|---|---|
| **Data drift** | `project.get_data_drift_dashboard(payload)` | Dashboard: which features moved, by statistical test (PSI, KS, Jensen–Shannon, …) |
| **Target drift** | `project.get_target_drift_dashboard(payload)` | Dashboard: has the outcome distribution changed |
| **Model performance** | `project.get_model_performance(model_name)` | Dashboard of the model's metrics |
| **Monitors** | `project.create_monitor(payload)` | A scheduled check (daily, weekly, …) that raises an alert when its threshold is crossed |
| **Alerts** | `project.alerts()` · `project.get_alert_details(id)` | Every alert a monitor has raised, with its detailed report |

Monitors run on their schedule, so a monitor created now raises its first alert on its next run; rerun section 9 later to see those alerts flow into the webhook. The dashboards below are generated immediately, and the retraining agent reads both.

In [ ]:
LAST_TAG = MONTH_TAGS[-1]
drift_payload = {"base_line_tag": ["Training"], "current_tag": [LAST_TAG], "stat_test_name": "psi", "stat_test_threshold": 0.2,
                 "features_to_use": [c for c in MI.index if c in FEATURES][:12]}
drift_dash = attempt(f"data drift dashboard · Training vs {LAST_TAG}", lambda: project.get_data_drift_dashboard(payload=drift_payload, pod=DASHBOARD_POD))
target_payload = {"base_line_tag": ["Training"], "current_tag": [LAST_TAG], "baseline_true_label": TARGET_COL, "current_true_label": TARGET_COL,
                  "model_type": TASK, "stat_test_name": "psi" if TASK == "classification" else "wasserstein", "stat_test_threshold": 0.1}
target_dash = attempt(f"target drift dashboard · Training vs {LAST_TAG}", lambda: project.get_target_drift_dashboard(payload=target_payload, pod=DASHBOARD_POD))

In [ ]:
# @title Drift dashboards from the platform, and drift by month { display-mode: "form" }
for name, dash in (("Data drift", drift_dash), ("Target drift", target_dash)):
    if dash is not None and hasattr(dash, "plot"):
        display(Markdown(f"**{name} dashboard from the platform**"))
        attempt(f"{name} dashboard", lambda d=dash: d.plot(height=520), show=False)
diag = attempt("data drift diagnosis", lambda: project.data_drift_diagnosis(baseline_tags=["Training"], current_tags=[LAST_TAG], pod=DASHBOARD_POD), show=False)
display(as_frame(diag).head(12))

# The same drift, month by month, computed locally from the uploaded files (the evidence the retraining agent reads)
MON_FEATURES = [c for c in MI.index if c in FEATURES and c in NUM][:12]
psi_tab = pd.DataFrame({t: {f: psi(frames["Training"][f], frames[t][f]) for f in MON_FEATURES} for t in MONTH_TAGS})
fig = go.Figure(go.Heatmap(z=psi_tab.values, x=psi_tab.columns, y=[c[:28] for c in psi_tab.index], colorscale=[[0, "#F7F7F4"], [0.4, GOLD], [1, RED]],
                           zmin=0, zmax=0.5, text=psi_tab.round(2).values, texttemplate="%{text}", colorbar=dict(title="PSI")))
fig.update_layout(template="plotly_white", height=460, title=f"Population stability by feature and month (above 0.2 = material drift) · drifting feature: {drift_feature}")
fig.show()

## 9.2 Monitors and alerts
A monitor is a scheduled check on the platform. When its threshold is crossed it raises an alert (and e-mails `ALERT_EMAILS`, if any are set).

In [ ]:
common = {"mail_list": ALERT_EMAILS, "frequency": "daily", "priority": 2, "base_line_tag": ["Training"], "current_tag": [LAST_TAG], "pod": DASHBOARD_POD}
MONITORS = [
    {"trigger_type": "Data Drift", "trigger_name": f"{MONITOR_PREFIX} data drift", **common, "stat_test_name": "psi", "stat_test_threshold": 0.2,
     "datadrift_features_per": 20, "dataset_drift_percentage": 50, "features_to_use": []},
    {"trigger_type": "Target Drift", "trigger_name": f"{MONITOR_PREFIX} target drift", **common, "model_type": TASK,
     "stat_test_name": target_payload["stat_test_name"], "stat_test_threshold": 0.1, "baseline_true_label": TARGET_COL, "current_true_label": TARGET_COL},
]
have = set(as_frame(attempt("monitors", lambda: project.monitors(), show=False)).get("trigger_name", []))
for payload in MONITORS:
    if payload["trigger_name"] not in have:
        attempt(f"create_monitor · {payload['trigger_name']}", lambda p=payload: project.create_monitor(dict(p)))
as_frame(attempt("monitors", lambda: project.monitors(), show=False))

In [ ]:
platform_alerts = as_frame(attempt("alerts raised so far", lambda: project.alerts(), show=False))
display(platform_alerts.head(10) if len(platform_alerts) else Markdown("*No platform alerts yet: the monitors above raise their first alerts on their next scheduled run.*"))

## 9.3 The retraining webhook and agent, on the platform
Same design as the open-source notebooks, with the platform doing the work:

- **Alerts** come from two places: the month-by-month checks below (feature drift from the uploaded tags, and the live model's score on each month, scored by the platform), and the platform's own monitors (`project.alerts()`).
- Every alert is POSTed, signed, to the **retraining webhook**. It rejects unsigned requests, ignores duplicates, logs warnings, and opens **one retraining job per month** for critical alerts.
- The **retraining agent** takes each job: **observe** → **decide** (its LLM reviews the evidence through the gateway) → **human gate** → **retrain** on the platform (`project.train_model` on the Training tag plus the labelled production months) → **verify** champion vs challenger on the next, unseen month, or on the held-out Test tag when there is no later month (`project.model_inference`) → **LLM review + human gate** → **promote** (`project.activate_model`).

`MAX_RETRAINS` caps the number of platform training jobs the agent may start in one run.

In [ ]:
# @title Webhook: the retraining agent's inbox (HTTP server, signature check, routing, job queue) and the alert rules engine { display-mode: "form" }
import hmac, hashlib, queue, threading, uuid, secrets as _secrets
import requests
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

WEBHOOK_SECRET = _secrets.token_hex(16)      # shared between the monitoring system and the webhook


class RetrainingWebhook:
    """HTTP endpoint that receives monitoring alerts and queues retraining jobs for the agent.

    POST /alerts   JSON alert, header X-Signature: sha256=<HMAC of the body>   → 202 queued | 200 logged | 401 | 422
    GET  /jobs     queued and finished jobs
    GET  /health
    """
    REQUIRED = ("alert_id", "rule", "monitor", "severity", "action", "period", "metric", "value", "threshold")

    def __init__(self, secret):
        self.secret, self.queue, self.jobs, self.seen, self.log = secret.encode(), queue.Queue(), {}, set(), []

    def _handle(self, raw, signature):
        expected = "sha256=" + hmac.new(self.secret, raw, hashlib.sha256).hexdigest()
        if not hmac.compare_digest(signature or "", expected):
            return 401, {"accepted": False, "reason": "invalid signature"}
        try:
            alert = json.loads(raw)
        except ValueError:
            return 422, {"accepted": False, "reason": "body is not JSON"}
        missing = [k for k in self.REQUIRED if k not in alert]
        if missing:
            return 422, {"accepted": False, "reason": f"missing fields: {missing}"}
        if alert["alert_id"] in self.seen:
            return 200, {"accepted": False, "reason": "duplicate alert (already received)"}
        self.seen.add(alert["alert_id"])
        if not (alert["severity"] == "critical" and alert["action"] == "retrain"):
            self.log.append({**alert, "routed_to": "logged / owner notified"})
            return 200, {"accepted": True, "routed_to": "logged / owner notified"}
        job = next((j for j in self.jobs.values() if j["period"] == alert["period"] and j["status"] == "queued"), None)
        if job is None:                                # one retraining job per period, however many alerts point at it
            job = {"job_id": "job-" + uuid.uuid4().hex[:6], "period": alert["period"], "status": "queued", "alerts": []}
            self.jobs[job["job_id"]] = job
            self.queue.put(job["job_id"])
        job["alerts"].append(alert)
        self.log.append({**alert, "routed_to": job["job_id"]})
        return 202, {"accepted": True, "routed_to": "retraining agent", "job_id": job["job_id"]}

    def start(self):
        hook = self

        class Handler(BaseHTTPRequestHandler):
            def log_message(self, *args):
                pass

            def _reply(self, code, body):
                data = json.dumps(body, default=str).encode()
                self.send_response(code); self.send_header("Content-Type", "application/json")
                self.send_header("Content-Length", str(len(data))); self.end_headers(); self.wfile.write(data)

            def do_POST(self):
                if self.path != "/alerts":
                    return self._reply(404, {"error": "not found"})
                raw = self.rfile.read(int(self.headers.get("Content-Length", 0)))
                self._reply(*hook._handle(raw, self.headers.get("X-Signature")))

            def do_GET(self):
                if self.path == "/health":
                    return self._reply(200, {"status": "ok", "queued": hook.queue.qsize()})
                if self.path == "/jobs":
                    return self._reply(200, list(hook.jobs.values()))
                self._reply(404, {"error": "not found"})

        self.server = ThreadingHTTPServer(("127.0.0.1", 0), Handler)       # port 0 = any free port
        threading.Thread(target=self.server.serve_forever, daemon=True).start()
        self.url = f"http://127.0.0.1:{self.server.server_address[1]}"
        return self

    def stop(self):
        self.server.shutdown()


def send_alert(url, alert, secret=WEBHOOK_SECRET):
    """What the monitoring system does: sign the alert and POST it."""
    body = json.dumps(alert, default=str).encode()
    sig = "sha256=" + hmac.new(secret.encode(), body, hashlib.sha256).hexdigest()
    return requests.post(url + "/alerts", data=body, headers={"Content-Type": "application/json", "X-Signature": sig}, timeout=10)


def evaluate_rules(metrics, rules, period, model_version):
    """Turn monitor metrics into alerts. A rule fires 'warning' above its warn level and 'critical' above its critical level."""
    alerts = []
    for r in rules:
        v = metrics.get(r["metric"])
        if v is None or not np.isfinite(v):
            continue
        sev = "critical" if v >= r["critical"] else "warning" if v >= r["warn"] else None
        if sev:
            alerts.append({"alert_id": f"{period}:{r['rule']}", "rule": r["rule"], "monitor": r["monitor"], "severity": sev,
                           "action": r["action"] if sev == "critical" else "notify", "period": period, "metric": r["metric"],
                           "value": round(float(v), 4), "threshold": r["critical"] if sev == "critical" else r["warn"],
                           "model_version": model_version, "raised_at": pd.Timestamp.now().isoformat(timespec="seconds")})
    return alerts

In [ ]:
ALERT_RULES = [   # warning → notify the owner · critical → the action
    dict(rule="feature drift",       monitor="data drift",  metric="max_feature_psi",        warn=0.10, critical=0.25, action="retrain"),
    dict(rule="many features drift", monitor="data drift",  metric="share_features_drifted", warn=0.15, critical=0.30, action="retrain"),
    dict(rule="performance decay",   monitor="performance", metric="performance_drop",       warn=0.02, critical=0.05, action="retrain"),
]
hook = RetrainingWebhook(WEBHOOK_SECRET).start()
requests.get(hook.url + "/health").json()

In [ ]:
# @title The retraining agent on the platform (definition) { display-mode: "form" }
PROMOTE_MARGIN = 0.002
PRED_COL = "pred_proba_AutoML" if TASK == "classification" else "Predicted_value_AutoML"


class PlatformRetrainingAgent:
    def __init__(self, webhook, live_build, live_label, release_score):
        self.webhook, self.live, self.label, self.release = webhook, live_build, live_label, release_score
        self.version, self.history, self.events, self._scores, self.retrains = 1, [], [], {}, 0

    def score(self, tag, build):
        """Score a tag on the platform (project.model_inference) and measure the result against the labels we hold."""
        if (tag, build) not in self._scores:
            res = project.model_inference(tag=tag, model_name=build, compute_type=COMPUTE)
            truth_ = dict(zip(frames[tag][UID].astype(str), frames[tag][TARGET_COL]))
            yt, s = res[UID].astype(str).map(truth_), res[PRED_COL].astype(float).to_numpy()
            self._scores[(tag, build)] = float(skm.roc_auc_score((yt.astype(str) == "1").astype(int), s) if TASK == "classification"
                                               else skm.r2_score(yt.astype(float), s))
        return self._scores[(tag, build)]

    def observe(self, tag):
        """Monitoring for one month: platform score of the live model, feature drift, and the alerts the rules raise."""
        live = attempt(f"{tag}: score with {self.live}", lambda: self.score(tag, self.live), show=False)
        f_psi = psi_tab[tag]
        m = {"max_feature_psi": float(f_psi.max()), "share_features_drifted": float((f_psi > 0.2).mean()),
             "performance_drop": (self.release - live) if live is not None else float("nan")}
        alerts = evaluate_rules(m, ALERT_RULES, tag, f"v{self.version}")
        self.history.append({"month": tag, "live_build": self.live, "live_score": live, **m, "alerts": len(alerts), "action": "none"})
        return alerts

    def run_pending(self):
        while not self.webhook.queue.empty():
            self.handle(self.webhook.jobs[self.webhook.queue.get()])

    def _mark(self, tag, action):
        for h in self.history:
            if h["month"] == tag:
                h["action"] = action

    def handle(self, job):
        tag = job["period"]; k = MONTH_TAGS.index(tag) if tag in MONTH_TAGS else len(MONTH_TAGS) - 1
        nxt = MONTH_TAGS[k + 1] if k + 1 < len(MONTH_TAGS) else "Test"          # no later month yet → verify on the held-out Test tag
        reasons = [f"{a['rule']} ({a['metric']} {a['value']} ≥ {a['threshold']})" for a in job["alerts"]]
        audit("retrain-agent", f"{job['job_id']} observe", f"{tag}: " + "; ".join(reasons))
        if self.retrains >= MAX_RETRAINS:
            why = f"MAX_RETRAINS = {MAX_RETRAINS} reached"
            job["status"] = f"deferred: {why}"; self._mark(tag, "deferred")
            audit("retrain-agent", f"{job['job_id']} decide", why, "DEFERRED")
            return
        verdict = agent_llm("retraining agent", f"{tag} decide", {
            "proposal": f"retrain the live model on the Training tag plus the labelled production months, then verify it on unseen data ({nxt})",
            "live_model": self.live, "metric": METRIC_KEY, "score_at_release": round(self.release, 4),
            "alerts": [{k_: a[k_] for k_ in ("rule", "severity", "metric", "value", "threshold")} for a in job["alerts"]],
            "platform_alerts": int(len(platform_alerts))})
        if not human_gate(f"{tag}: retrain because {'; '.join(reasons)}? (LLM reviewer: {verdict['decision']})", verdict["decision"] == "approve"):
            job["status"] = "rejected by reviewer"; self._mark(tag, "retraining rejected")
            return
        tags = ["Training"] + MONTH_TAGS[: k + 1]
        base = self.label if self.label in ROSTER else ROSTER[0]
        self.retrains += 1
        challenger = train(project, f"Challenger · {base} on {' + '.join(tags)}", model=base, training_strategy="simple", tunning_strategy="inference",
                           compute_type=COMPUTE, model_config=model_config_for(base), data_config={**base_data_config(xai_for(base, TASK)), "tags": tags})
        c_old = attempt(f"{nxt}: champion", lambda: self.score(nxt, self.live), show=False) if challenger else None
        c_new = attempt(f"{nxt}: challenger", lambda: self.score(nxt, challenger), show=False) if challenger else None
        if c_old is None or c_new is None:
            job["status"] = "challenger could not be trained or scored"; self._mark(tag, "retraining failed")
            return
        audit("retrain-agent", f"{job['job_id']} verify", f"on unseen {nxt}: champion {c_old:.4f} vs challenger {c_new:.4f} {METRIC_KEY}")
        win = c_new - c_old >= PROMOTE_MARGIN
        v2 = agent_llm("retraining agent", f"{tag} promote", {
            "proposal": "promote the challenger to live", "verified_on": nxt, "champion_score": round(c_old, 4), "challenger_score": round(c_new, 4),
            "margin_required": PROMOTE_MARGIN, "clears_margin": bool(win)}, default="approve" if win else "reject")
        promoted = bool(win and human_gate(f"Promote {challenger} ({c_new - c_old:+.4f} {METRIC_KEY}; LLM reviewer: {v2['decision']})?", v2["decision"] == "approve"))
        if promoted:
            attempt(f"activate {challenger}", lambda: ensure_inference(project, challenger), show=False)
            self.live, self.version = challenger, self.version + 1
        audit("retrain-agent", f"{job['job_id']} promote", f"margin required {PROMOTE_MARGIN}", f"PROMOTED {challenger} (v{self.version})" if promoted else f"KEPT {self.live}")
        job.update(status="promoted" if promoted else "kept champion", champion_next=c_old, challenger_next=c_new)
        self._mark(tag, "promoted challenger" if promoted else "retrained, kept champion")
        self.events.append({"job_id": job["job_id"], "month": tag, "reasons": reasons, "tags": tags, "challenger": challenger, "next_month": nxt,
                            "champion_next": c_old, "challenger_next": c_new, "promoted": promoted})


def forward_platform_alerts(alerts_frame, url):
    """Alerts raised by the platform's own monitors go to the same webhook."""
    for i, a in alerts_frame.head(20).iterrows():
        a = a.to_dict()
        period = next((t for t in MONTH_TAGS if t in json.dumps(a, default=str)), LAST_TAG)
        send_alert(url, {"alert_id": f"platform:{a.get('id', a.get('_id', i))}", "rule": str(a.get("trigger_name", "platform monitor")),
                         "monitor": str(a.get("trigger_type", "platform monitor")), "severity": "critical", "action": "retrain", "period": period,
                         "metric": "platform monitor", "value": 1, "threshold": 1, "source": "Lexsi platform monitor"})

In [ ]:
retrain_agent = PlatformRetrainingAgent(hook, CHAMPION_BUILD, CHAMPION, release_score=float(perf["roc_auc" if TASK == "classification" else "r2"]))
for tag in MONTH_TAGS:
    for alert in retrain_agent.observe(tag):                       # platform scoring + drift for this month → alerts
        send_alert(hook.url, alert)                                # monitoring → webhook (signed POST)
    retrain_agent.run_pending()                                    # the agent works through whatever the webhook queued
forward_platform_alerts(platform_alerts, hook.url)                 # alerts raised by the platform's own monitors → the same webhook
retrain_agent.run_pending()
hook.stop()
pd.DataFrame(retrain_agent.history).drop(columns=["live_build"])

In [ ]:
# @title Months in production: score, drift, alerts and the retraining outcome { display-mode: "form" }
H = pd.DataFrame(retrain_agent.history)
fig = make_subplots(rows=1, cols=3, column_widths=[0.36, 0.32, 0.32], subplot_titles=(f"{METRIC_KEY} by month · live model, scored by the platform",
                                                                                     "Strongest feature drift by month (PSI)", "Champion vs challenger on unseen data"))
fig.add_trace(go.Scatter(x=H.month, y=H.live_score, mode="lines+markers", line=dict(color=BLUE, width=3), name="live model"), 1, 1)
fig.add_trace(go.Scatter(x=H.month, y=[retrain_agent.release] * len(H), mode="lines", line=dict(color=TEAL, dash="dot"), name="release level"), 1, 1)
fig.add_trace(go.Bar(x=H.month, y=H.max_feature_psi, marker_color=[RED if v >= 0.25 else GOLD if v >= 0.10 else GREY for v in H.max_feature_psi], showlegend=False), 1, 2)
for e in retrain_agent.events:
    fig.add_trace(go.Bar(x=[f"{e['next_month']} champion", f"{e['next_month']} challenger"], y=[e["champion_next"], e["challenger_next"]], marker_color=[GREY, BLUE],
                         showlegend=False, text=[f"{e['champion_next']:.4f}", f"{e['challenger_next']:.4f}"], textposition="outside"), 1, 3)
fig.update_layout(template="plotly_white", height=430, legend=dict(orientation="h", y=-0.15),
                  title=f"Monitoring and retraining on the platform · live model: {retrain_agent.live} (v{retrain_agent.version})")
fig.show()
log = pd.DataFrame(hook.log)
if len(log):
    display(log[["period", "rule", "severity", "value", "threshold", "routed_to"]].style.hide(axis="index").set_caption("Every alert the webhook received, and where it was routed"))
display(pd.DataFrame([{k: v for k, v in j.items() if k != "alerts"} | {"alerts": len(j["alerts"])} for j in hook.jobs.values()]))

## 9.4 Model card, regenerated
The facts now include monitoring and the retraining outcome. The card is written again through the gateway, with a change log.

In [ ]:
# @title Facts added since version 1 { display-mode: "form" }
record("monitoring", months_monitored=len(retrain_agent.history), drift_feature=drift_feature, max_feature_psi=float(psi_tab.values.max()),
       platform_monitors=len(as_frame(attempt("monitors", lambda: project.monitors(), show=False))), platform_alerts=int(len(platform_alerts)),
       alerts_received_by_webhook=len(hook.log), retraining_jobs_opened=len(hook.jobs),
       alert_rules="; ".join(f"{r['rule']}: warning at {r['warn']}, critical at {r['critical']}" for r in ALERT_RULES), promotion_margin=PROMOTE_MARGIN,
       plan="platform drift dashboards and scheduled monitors; monthly drift and performance checks raise alerts; critical alerts go to a signed "
            "retraining webhook; an agent retrains on the platform, verifies on unseen data and promotes only after LLM review and human approval")
if retrain_agent.events:
    e = retrain_agent.events[0]
    record("retraining", trigger_month=e["month"], trigger="; ".join(e["reasons"]), retraining_tags=", ".join(e["tags"]), challenger_build=e["challenger"],
           verified_on=e["next_month"], champion_score_on_unseen_data=e["champion_next"], challenger_score_on_unseen_data=e["challenger_next"],
           retraining_runs=len(retrain_agent.events),
           outcome="challenger promoted" if e["promoted"] else "champion kept", live_platform_model=retrain_agent.live,
           human_approvals="retraining and promotion gates, each after an LLM review")
    changes = [f"{e['month']}: retraining triggered ({'; '.join(e['reasons'])}); challenger {e['challenger']} trained on tags {', '.join(e['tags'])}.",
               f"On unseen {e['next_month']} the champion scored {e['champion_next']:.4f} and the challenger {e['challenger_next']:.4f}; "
               + ("challenger promoted after human approval." if e["promoted"] else "champion kept, the challenger did not clear the promotion margin.")]
else:
    record("retraining", outcome="no retraining was completed during the monitoring period", live_platform_model=retrain_agent.live)
    changes = ["Monitoring period completed with no retraining."]
FACTS["use_case"]["version"] = f"v{retrain_agent.version}"
FACTS["governance"]["human_approval_gates"] = sum(1 for a in AUDIT if a["actor"] == "human")
FACTS["governance"]["llm_reviewed_agent_steps"] = sum(1 for t in AGENT_TRACE if t["decision"] in ("approve", "reject"))
for c in changes:
    print("•", c)

In [ ]:
CARD_V2, TRACE_V2, CHECKS_V2 = generate_model_card(FACTS, changes=changes, version_label="v2")

In [ ]:
show_card(CARD_V2, TRACE_V2, CHECKS_V2)
open("model_card_v2.md", "w").write(CARD_V2);

# 10 · Agent traces, conclusions and audit trail

## 10.1 Traces of every agent
Three agents ran in this notebook, and each one used an LLM through the Lexsi gateway:

| Agent | What it decided | Role in `GATEWAY_MODELS` |
|---|---|---|
| **Data-prep agent** | which features to remove, step by step | `agents` |
| **Retraining agent** | whether to retrain, and whether to promote the challenger | `agents` |
| **Model-card writer** | the model card and its revision | `card_writer` |

The deterministic checks (drift, leakage probes, scores) produce the evidence; the LLM reviews it and explains its decision; a human gate confirms. The first table is this notebook's own log of every LLM call. The second is read back from the platform: the gateway stores each call as a session message with a trace (input, guardrails, LLM, output).

In [ ]:
# @title Every LLM call, by agent { display-mode: "form" }
T = pd.DataFrame(AGENT_TRACE)
if len(T):
    by = T.groupby("agent").agg(calls=("agent", "size"), seconds=("seconds", "sum"), input_tokens=("input_tokens", "sum"), output_tokens=("output_tokens", "sum"))
    fig = make_subplots(rows=1, cols=3, subplot_titles=("LLM calls per agent", "Seconds per agent", "Tokens per agent"))
    fig.add_trace(go.Bar(x=by.index, y=by.calls, marker_color=BLUE, showlegend=False), 1, 1)
    fig.add_trace(go.Bar(x=by.index, y=by.seconds, marker_color=TEAL, showlegend=False), 1, 2)
    fig.add_trace(go.Bar(x=by.index, y=by.input_tokens, name="input", marker_color=GREY), 1, 3)
    fig.add_trace(go.Bar(x=by.index, y=by.output_tokens, name="output", marker_color=ORANGE), 1, 3)
    fig.update_layout(template="plotly_white", height=380, barmode="stack", title=f"Agent activity · {len(T)} LLM calls through the Lexsi gateway · models: {', '.join(sorted(set(T.model)))}")
    fig.show()
    display(T[["at", "agent", "step", "model", "via", "seconds", "input_tokens", "output_tokens", "decision", "rationale"]]
            .style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "pre-wrap"}).set_caption("Every LLM call made by an agent in this run"))
else:
    print("No LLM calls were made (no gateway configured): the agents used their rule defaults.")

In [ ]:
sessions = as_frame(attempt("gateway sessions", lambda: text_project.sessions(), show=False)) if text_project is not None else pd.DataFrame()
session_id = _SESSION["id"] if (_SESSION["use"] and text_project is not None) else (str(sessions.iloc[0].get("session_id", "")) if len(sessions) else "")
messages = as_frame(attempt("messages in this run's session", lambda: text_project.messages(session_id), show=False)) if session_id else pd.DataFrame()
display(sessions.head(5)); display(messages.head(20))

In [ ]:
# @title One call, step by step, from the gateway's trace { display-mode: "form" }
if len(messages) and "trace_id" in messages:
    last_trace = str(messages["trace_id"].iloc[-1])
    tr = as_frame(attempt(f"trace {last_trace}", lambda: text_project.traces(last_trace), show=False))
    keep = [c for c in ("component", "parent_component", "duration", "created_at") if c in tr]
    display(tr[keep] if keep else tr.head(10))
    if {"component", "duration"} <= set(tr.columns):
        fig = px.bar(tr, x="duration", y="component", orientation="h", color_discrete_sequence=[BLUE], title="One LLM call, step by step, as traced by the gateway (seconds)")
        fig.update_layout(template="plotly_white", height=320, yaxis=dict(autorange="reversed")); fig.show()
else:
    display(Markdown("*Platform traces are listed in the Lexsi console under the gateway project → Sessions. They could not be read back here.*"))

## 10.2 Conclusions and audit trail

In [ ]:
# @title Conclusions in plain English { display-mode: "form" }
P_, S, U = FACTS["performance"], FACTS["selection"], FACTS["uncertainty"]
headline = (f"ROC AUC {P_.get('roc_auc', float('nan')):.3f} at a threshold of {P_.get('threshold', float('nan')):.2f}" if TASK == "classification"
            else f"R² {P_.get('r2', float('nan')):.3f}")
T = pd.DataFrame(AGENT_TRACE)
CONCLUSIONS = [
    f"**Data:** {FACTS['preparation']['features_before']:.0f} candidate features became {FACTS['features']['final_feature_count']:.0f} model inputs after the data-prep agent and feature engineering; each agent step was " + ("reviewed by an LLM and " if len(T) else "") + "confirmed at a human gate.",
    f"**Model:** {S['candidates_succeeded']:.0f} builds were trained on the Lexsi platform; **{S['selected_model']}** ({S['platform_model_name']}) was selected on the Test tag ({S['selection_metric']} {S['selected_test_tag_score']:.3f}).",
    f"**Performance on unseen data:** {headline}, scored by the platform.",
    f"**Uncertainty:** a {U['target_coverage']:.0%} coverage promise measured {U['measured_coverage']:.1%}.",
    f"**Guardrails:** observations on the platform: {FACTS['governance']['platform_observations']:.0f}; policies: {FACTS['governance']['platform_policies']:.0f}. They are evaluated on every case.",
    f"**Monitoring:** drift dashboards and {FACTS['monitoring']['platform_monitors']:.0f} scheduled monitors on the platform; {FACTS['monitoring']['alerts_received_by_webhook']:.0f} alerts reached the webhook; retraining outcome: {FACTS['retraining']['outcome']}.",
    (f"**Agents:** {len(T)} LLM calls through the Lexsi gateway using {', '.join(sorted(set(T.model)))}; every call is traced in the gateway project."
     if len(T) else "**Agents:** no LLM gateway was configured, so the agents used their rule defaults."),
    f"**Documentation:** the model card was written by {TRACE_V2['writer']}; {CHECKS_V2['grounded_pct']:.0f}% of its numbers trace back to recorded facts.",
]

In [ ]:
# @title Summary and the full audit trail { display-mode: "form" }
display(Markdown("## What this exercise showed\n\n" + "\n".join(f"- {line}" for line in CONCLUSIONS)))
trail = pd.DataFrame(AUDIT)
if len(trail):
    fig = px.histogram(trail, x="step", color="actor", color_discrete_sequence=PALETTE, title="Audit trail · logged agent steps and human decisions")
    fig.update_layout(template="plotly_white", height=360, xaxis_title=""); fig.show()
    display(trail.style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))

In [ ]:
# @title Platform run log: every training call and the model registry { display-mode: "form" }
display(pd.DataFrame(RUNS))
with tolerant_drop():
    display(project.models()[["model_name", "status", "tabular_model_type", "instance_type", "created_at"]]
            .sort_values("created_at", ascending=False).head(20))